<a href="https://colab.research.google.com/github/farahdinaariff/Data-Science-Project/blob/main/TropicalPV_Probabilistic_Solar_Forecasting_v9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TropicalPV — Research Evolution v5 → v6 → v7 → v7.1
## A complete research-development notebook: what was built, what the evidence showed, what was changed, and why

**Farah Dina Bte Esa Mahmud Ariff** · MSc Data Science (University of Hertfordshire / PSB Academy) · 7FTC2001

---

### 1. Purpose

This notebook documents the **research trail** of the TropicalPV probabilistic solar-forecasting project across four successive pipeline versions. It is not four notebooks pasted together. It follows the recurring researcher's loop:

**implementation → observation → problem → investigation → modification → validation → next observation**

For every change it shows the evidence that prompted it, the before/after result, and the consequence for interpretation.

### How evidence is produced here: three labelled sources

| Label | Meaning | Why |
|---|---|---|
| 🔬 **RECALCULATED** | Computed in this notebook from your raw data files | The evidence for every *decision*: timing, QC, features, leakage, controlled experiments |
| 📓 **REPRODUCED** | Tables and figures extracted verbatim from the executed v5 / v6 / v7.1 notebooks on your Drive | The full-scale results of each version, which took about 10 GPU-hours in total and are not re-trained here |
| 📌 **RECORDED** | Values copied from those executed outputs, with the cell of origin | Headline numbers for the comparison tables |

Each consequence of a methodological problem is classified as **Demonstrated** (shown by data in this notebook) or **Potential** (plausible, but not directly shown).

**Runtime (Colab GPU):** about 20–30 min on the first run. The Darwin 1-second ingestion (~90 M rows) dominates and is cached to Drive, so re-runs take about 10 min.

### 2. Research context

The project forecasts the **clear-sky index** k_t = GHI / GHI_clear-sky one hour ahead at **Darwin** (tropical savanna, Köppen Aw). It compares two deterministic architectures (LSTM, GRU) with two uncertainty-quantifying architectures (MC-Dropout LSTM, Quantile-Regression GRU). Irradiance stands in for PV power, a proxy that is tested at **Alice Springs**. Transfer is evaluated at further sites.

Working with k_t has a precise requirement: **measured GHI and the computed clear-sky GHI must refer to the same instant**. Much of the research trail below concerns exactly that requirement.

### 3. Research question and objectives (approved proposal / PDM)

> **RQ.** Which deep learning architecture — deterministic (LSTM, GRU) or probabilistic (MC-Dropout LSTM, Quantile Regression GRU) — produces the most accurate and best-calibrated solar irradiance forecasts for tropical climates under wet/dry seasonal conditioning?

| Objective | Requirement |
|---|---|
| O1 | Accuracy (RMSE/MAE/R²) and skill vs smart persistence; secondary matched irradiance-vs-PV test at Alice Springs |
| O2 | Calibration (CRPS, pinball, PICP, MIW); does UQ trade off against accuracy under wet-season variability? |
| O3 | Binary wet/dry conditioning; ramp events (Δk_t, elevation > 15°); direction and magnitude reported |
| O4 | Decision support: PICP at nominal and MIW ≤ X |
| PDM O8 | Cross-climate validation |

### 4. Version history and inventory of analytical work

The four executed notebooks were inspected cell by cell. Each substantial component is listed with the action taken here: **KEEP** (recalculated in this notebook), **REPRODUCE** (its executed output is shown), **CONSOLIDATE** (merged with a later equivalent), **MODIFY**, or **REMOVED** (with the reason).

| Component (Part) | v5 | v6 | v7 | v7.1 | Action here |
|---|---|---|---|---|---|
| Data audit / decision rule (1) | ✓ | ✓ + raw inventory (1.0) | ✓ + design record (1.D), licences (1.E) | = v7 | REPRODUCE (tables in §16); design record in §15 |
| Streaming ingestion of 1-s data (2–3) | ✓ Local column | ✓ Local column + bin offset | ✓ **UTC column** + offset check | = v7 | **KEEP**: one pass builds all three treatments (§5.1) |
| Data validation, timing lag (3A) | — | ✓ (±45 min) | ✓ clear-day gate ±10 min | = v7 | **KEEP** (§5.2, §7, §11) |
| QC + k_t + gap interpolation (4) | ✓ | ✓ | ✓ + *_obs covariates | = v7 | **KEEP** for every version (§5.4, §6, §7) |
| Feature engineering (5.1) | ✓ 19 candidates → screen | ✓ | ✓ endogenous by design | = v7 | **KEEP** (§5.6, §10) |
| EDA, Figures 1–10 (5) | ✓ | ✓ | ✓ | ✓ | **KEEP** for Darwin/Alice; REPRODUCE all-site figures (§9) |
| Energy regimes, ramps, VI (5A) | ✓ | ✓ | ✓ VI replaced; shares fixed | = v7 | **KEEP** + demonstrate both defects (§9) |
| Supervised windows, split (6) | 70/15/15 | 70/15/15 | 60/12.5/12.5/15 + rolling origin | = v7 | **KEEP** (§5.7, §7) |
| Four architectures (7) | ✓ | ✓ | ✓ + HMC-LSTM, tuned | = v7 | **KEEP** in controlled experiments (§6–7) |
| Metrics, naive baselines (8, 8A) | ✓ | ✓ | ✓ + CRPS_q, Winkler | = v7 | **KEEP** |
| Lookback study (8B) | ✓ | ✓ | diagnostic only; tuning 8C | = v7 | REPRODUCE (Fig 24) |
| Main results, calibration, residuals (9–9B) | ✓ | ✓ | last origin (descriptive) | = v7 | REPRODUCE per version (§5.8, §6, §7) |
| Seasonal ablation (10) | single-seed, confounded | same | **3-arm R3** | = v7 | **KEEP** the confound demonstration; REPRODUCE R3 |
| Ramp extremes (10A) | ✓ | ✓ | ✓ | ✓ | REPRODUCE |
| Unmatched proxy test (11) | ✓ | ✓ | **REMOVED** | — | REMOVED: superseded by the matched test; its number leaked into v6's findings |
| Matched proxy test (11A → 11) | ✓ | ✓ (row-deletion windows) | PV/capacity target | **PV clear-sky index** | **KEEP** + demonstrate both defects (§8) |
| Cross-climate (12) | 4 sites | 7 sites | 8 sites + conformal transfer + shift | = v7 | REPRODUCE |
| Robustness: seeds, DM, horizon (12A) | ✓ | ✓ | moved into Part R | = v7 | **KEEP** the DM demonstration (§7) |
| Decision support, reserve (13, 13A) | raw | raw | conformal, equal reliability | = v7 | REPRODUCE |
| Part R robust protocol | — | — | ✓ | ✓ | REPRODUCE (§16) |
| Findings / leaderboard (14, 14A, 14B) | ✓ | ✓ (contradictory values) | regenerated from R | ✓ | CONSOLIDATE (§16–19) |
| Timestamp comparison notebook (v8) | — | — | — | separate | CONSOLIDATE into §11 |

---
## 0. Setup: packages, paths and discovery of files

**What.** Mount Drive, locate every raw data file and the **executed** v5, v6, v7 and v7.1 notebooks, and create a cache folder.
**Why.** Everything else depends on these paths. The executed notebooks are needed for the 📓 REPRODUCED evidence. If any is missing, those cells say so and the rest still runs.

Copy the executed notebooks (the versions *with outputs*) anywhere under `DATA_ROOT`, e.g. `MyDrive/TropicalPV_notebooks/`.

In [ ]:
import os, sys, glob, json, time, re, math, platform, warnings, base64, io
import numpy as np, pandas as pd
import matplotlib, matplotlib.pyplot as plt
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 90)

IN_COLAB = os.path.exists("/content")
if IN_COLAB and not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

DATA_ROOT = os.environ.get("TROPICALPV_ROOT", "/content/drive/MyDrive")
CACHE = os.path.join(DATA_ROOT, "_tropicalpv_evolution"); os.makedirs(CACHE, exist_ok=True)
SEED = 42
np.random.seed(SEED)

# ---- site constants (identical in v5, v6, v7, v7.1: SITES registry) ----
DARWIN = dict(lat=-12.4256, lon=130.8916, tz=9.5, wet_months=[11, 12, 1, 2, 3, 4])
ALICE  = dict(lat=-23.7615, lon=133.8747, tz=9.5, wet_months=[12, 1, 2])
DARWIN_YEARS = ["2021", "2022", "2023"]            # the proposal's Darwin sample (v5, v6)

# ---- pipeline constants (CFG in v5-v7.1; unchanged unless stated) ----
KT_MAX, GHI_MAX, NEG_TOL, DAYLIGHT_CS_MIN = 1.30, 1400.0, -20.0, 20.0
MAX_INTERP_GAP_H, RAMP_KT, ELEV_FLOOR = 3, 0.35, 15.0
LOOKBACK, HORIZON, UNITS, EPOCHS, PATIENCE, BATCH, LR = 24, 1, 64, int(os.environ.get("EVO_EPOCHS", 60)), 8, 128, 1e-3
QUANTILES = [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
I_LO, I_HI = QUANTILES.index(0.10), QUANTILES.index(0.90)
print(f"DATA_ROOT = {DATA_ROOT}\nCACHE     = {CACHE}")

In [ ]:
# ---- one recursive scan of the data root; duplicates (same file name in two folders) keep the first ----
t0 = time.time()
ALL_FILES = [p for p in glob.glob(os.path.join(DATA_ROOT, "**", "*"), recursive=True) if os.path.isfile(p)]
print(f"scanned {len(ALL_FILES):,} files in {time.time()-t0:.0f}s")

def find(pattern, must=None, exclude=None):
    rx = re.compile(pattern, re.I)
    hits, seen = [], set()
    for p in sorted(ALL_FILES):
        b = os.path.basename(p)
        if rx.search(b) and (must is None or re.search(must, p, re.I)) and not (exclude and re.search(exclude, b, re.I)):
            if b not in seen:
                seen.add(b); hits.append(p)
    return hits

FILES = {
    "darwin_1s":   [p for p in find(r"fsm1m_7127.*darwin.*irradiancedata_1sec_(\d{4})\.csv$")
                    if re.search(r"_1sec_(\d{4})", p).group(1) in DARWIN_YEARS],
    "darwin_1min": find(r"fsm1m_7127.*darwin.*irradiancedata\.csv$"),
    "darwin_met":  find(r"fsm1m_7127.*darwin.*metdata\.csv$"),
    "alice_irr":   find(r"fsm1m_7111.*alice.*irradiancedata.*\.csv$"),
    "alice_pv":    find(r"^241-Site_DKA_Totals-PV.*\.csv$"),
    "sg_nasa":     find(r"^POWER_Point_Hourly_.*_LST.*\.csv$"),
    "sg_openmeteo": find(r"singapore.*\.csv$", must=r"open-?meteo", exclude=r"availability"),
}
NOTEBOOKS = {}
for v, pat in (("v5", r"TropicalPV_Probabilistic_Solar_Forecasting_v5\.ipynb$"),
               ("v6", r"TropicalPV_Probabilistic_Solar_Forecasting_v6\.ipynb$"),
               ("v7", r"TropicalPV_Probabilistic_Solar_Forecasting_v7\.ipynb$"),
               ("v7.1", r"TropicalPV_Probabilistic_Solar_Forecasting_v7_1\.ipynb$")):
    h = find(pat)
    NOTEBOOKS[v] = h[0] if h else None
REGISTRY_FILE = (find(r"^results_registry\.json$") or [None])[0]
WORKBOOK_FILE = (find(r"^TropicalPV_v7_results\.xlsx$") or [None])[0]

display(pd.DataFrame([{"input": k, "files": len(v), "first": os.path.basename(v[0]) if v else "— NOT FOUND —"}
                      for k, v in FILES.items()]))
display(pd.DataFrame([{"executed notebook": k, "found": bool(v), "path": v} for k, v in NOTEBOOKS.items()]
                     + [{"executed notebook": "v7.1 results_registry.json", "found": bool(REGISTRY_FILE), "path": REGISTRY_FILE},
                        {"executed notebook": "v7.1 results workbook", "found": bool(WORKBOOK_FILE), "path": WORKBOOK_FILE}]))
assert FILES["darwin_1s"], "Darwin 1-s files are required (the whole research trail starts from them)."

### 0.1 Evidence infrastructure: reading the executed notebooks (📓 REPRODUCED)

**What.** `show_reproduced(version, figure_name)` finds the cell of an executed notebook that saved `figure_name` and displays the image **exactly as that version produced it**. `executed_text(version, marker)` returns the printed output of the cell whose source contains `marker`.
**Why.** This preserves every version's results and figures as evidence without re-running roughly 10 GPU-hours of training. Nothing is re-drawn or edited.

In [ ]:
from IPython.display import Image as _IPImage
_NB_CACHE = {}

def _nb(version):
    p = NOTEBOOKS.get(version)
    if not p:
        return None
    if version not in _NB_CACHE:
        _NB_CACHE[version] = json.load(open(p))
    return _NB_CACHE[version]

def show_reproduced(version, figname, width=1000):
    nb = _nb(version)
    if nb is None:
        print(f"📓 {version} executed notebook not found — figure '{figname}' cannot be reproduced."); return
    for c in nb["cells"]:
        if c["cell_type"] != "code":
            continue
        src = "".join(c["source"])
        names = re.findall(r'savefig\(fig,\s*"([^"]+)"', src)
        if figname in names:
            imgs = [o["data"]["image/png"] for o in c.get("outputs", []) if "image/png" in o.get("data", {})]
            k = names.index(figname)
            if k < len(imgs):
                print(f"📓 REPRODUCED from executed {version}: {figname}")
                display(_IPImage(data=base64.b64decode(imgs[k]), width=width)); return
    print(f"📓 {figname} not found in executed {version} (not produced by that version).")

def executed_text(version, marker, maxchars=6000):
    nb = _nb(version)
    if nb is None:
        return f"(executed {version} notebook not found)"
    for c in nb["cells"]:
        if c["cell_type"] == "code" and marker in "".join(c["source"]):
            out = []
            for o in c.get("outputs", []):
                if o["output_type"] == "stream":
                    out.append("".join(o["text"]))
            return "".join(out)[-maxchars:]
    return f"(marker not found in {version})"

REGISTRY = json.load(open(REGISTRY_FILE)) if REGISTRY_FILE else {}
def reg(key, default=None):
    return REGISTRY.get(key, {}).get("value", default)
print("executed notebooks available:", [v for v, p in NOTEBOOKS.items() if p])
print("v7.1 registry entries:", len(REGISTRY))

### 0.2 📌 RECORDED headline values of each version (provenance given)

These are the values each version reported in its own executed output. They are kept as a table so later sections can compare them.

**Important.** v6 → v7 also changed the protocol and extended the Darwin record, so differences in that step are **not attributable to timing alone** (see §7).

In [ ]:
RECORDED = pd.DataFrame([
 # quantity, v5, v6, v7/v7.1, provenance
 ("Darwin test windows (single test block)", 535, 535, 4887, "v5/v6 Part 6 split log; v7 R1 fold table (4 rolling origins)"),
 ("Persistence RMSE, Darwin t+1 (W/m²)", 146.80, 129.31, 99.2, "v5/v6 Part 8A/9 tables; v7 R2 leaderboard (different test period)"),
 ("QR-GRU RMSE (W/m²)", 114.83, 120.30, 87.2, "Part 9 main table (seed 42); v7 R2 (20 runs)"),
 ("QR-GRU skill vs persistence (%)", 21.8, 7.0, 12.1, "Part 9 / 12A (seed 42); v7 R2 with 95% CI [10.6, 13.6]"),
 ("QR-GRU skill, seed mean (%)", 17.1, 2.9, 12.1, "Part 12A (3 seeds); v7 R2 (5 seeds x 4 folds)"),
 ("DM p, QR-GRU vs persistence", "<0.001", 0.134, "<0.001 (HAC, Holm)", "Part 12A (no HAC); v7 R2 (Newey-West + Holm)"),
 ("PICP80 MC-Dropout (raw)", 0.647, 0.563, 0.621, "Part 9A / R2"),
 ("PICP80 QR-GRU (raw)", 0.784, 0.727, 0.805, "Part 9A / R2"),
 ("Ramp RMSE penalty, mean (%)", 14.9, 2.7, "121-173 (per model)", "Part 10A / R5(c)"),
 ("Proxy delta, irradiance - PV (pp)", 6.51, 11.78, "-18.69 (v7) / -8.02 (v7.1)", "Part 11A; v7 & v7.1 Part 11"),
 ("Reserve, raw probabilistic vs deterministic (%)", -17.2, 40.3, "-2.5 (QR-GRU median)", "Part 13A; v7 R5(e)"),
 ("Singapore test windows", 8572, 51, "NASA POWER (5-fold larger record)", "Part 12 logs; v7 design log"),
 ("Darwin diurnal lag (min)", "not measured", -30.98, -1.29, "v6 Part 3A (all days); v7 registry (clear days)"),
], columns=["quantity", "v5", "v6", "v7 / v7.1", "provenance (executed output)"])
if REGISTRY:
    RECORDED.loc[RECORDED.quantity == "Darwin diurnal lag (min)", "v7 / v7.1"] = reg("timing.primary_clear_day_lag_min", -1.29)
display(RECORDED)

---
# 5. Version 5 — Baseline research implementation

v5 established the complete pipeline:

* streaming ingestion of the Darwin 1-second reference cells;
* QC and the clear-sky index;
* 19 candidate features with a screen;
* EDA (Figures 1–10) and energy-system diagnostics (5A);
* a 70/15/15 chronological split;
* four architectures, naive baselines, calibration, seasonal ablation, proxy, cross-climate and decision-support analyses.

This section **re-runs the v5 data pipeline** on your raw files, then reproduces v5's modelling results.

## 5.1 Data loading: one streaming pass that preserves every version's view

**What.** The three Darwin 1-second files (2021–2023, the proposal's sample, ~90 M rows) are read **once**. Each sample is accumulated into hourly bins under **two** time bases at the same time:

* **Local basis**, from `Timestamp_Local`. This is what v5 and v6 used.
* **ACST basis**, from `Timestamp_UTC` + 9 h 30 min. This is what v7 and v7.1 use.

For each bin the pass keeps the GHI sum and count and the mean within-bin sample offset, which v6 needs. For the local basis it also keeps the reference-cell temperature (v5's `temp_c`).

It also records file-level facts for validation (timestamp offsets, duplicates, gaps, sentinels) and keeps the raw 1-second rows of **July 2022** for the physical timing test in §11.

**Why one pass.** Reading 90 M rows several times would multiply the runtime. The result is cached to Drive.

In [ ]:
import hashlib
CACHE_KEY = hashlib.md5("|".join(f"{os.path.basename(p)}-{os.path.getsize(p)}" for p in FILES["darwin_1s"]).encode()).hexdigest()[:10]
P_LOCAL, P_ACST = f"{CACHE}/darwin_hourly_local_{CACHE_KEY}.parquet", f"{CACHE}/darwin_hourly_acst_{CACHE_KEY}.parquet"
P_JULY, P_FACTS = f"{CACHE}/darwin_1s_july2022_{CACHE_KEY}.parquet", f"{CACHE}/darwin_ingest_facts_{CACHE_KEY}.json"
CHUNK = 1_000_000
JULY = (pd.Timestamp("2022-07-01"), pd.Timestamp("2022-08-01"))          # UTC window kept at 1-s resolution

def _header_row(path):
    with open(path, "r", encoding="utf-8-sig", errors="replace") as f:
        for i, line in enumerate(f):
            if "timestamp_utc" in line.lower():
                return i
            if i > 50:
                raise ValueError(f"no Timestamp_UTC header in {path}")

def _fmt(sample):
    for f in ("%d/%m/%Y %H:%M:%S", "%d/%m/%Y %H:%M"):
        if pd.to_datetime(pd.Series(sample), format=f, errors="coerce").notna().mean() > 0.9:
            return f
    raise ValueError("unknown timestamp format")

def ingest_darwin_1s(files):
    acc_local, acc_acst, july, facts = [], [], [], []
    for path in files:
        t0 = time.time()
        hdr = _header_row(path)
        cols = pd.read_csv(path, skiprows=hdr, nrows=0).columns.tolist()
        cu = next(c for c in cols if c.lower().startswith("timestamp_utc"))
        cl = next(c for c in cols if c.lower().startswith("timestamp_local"))
        irr = [c for c in cols if c.lower().startswith("irradiance_mb")]
        tmp = [c for c in cols if c.lower().startswith("unittemp_mb")]
        fmt = _fmt(pd.read_csv(path, skiprows=hdr, usecols=[cu], nrows=200, dtype=str)[cu])
        rows = bad_ts = nonmono = gaps = sent = 0
        offsets, last_t = {}, None
        for ch in pd.read_csv(path, skiprows=hdr, usecols=[cu, cl] + irr + tmp, dtype={cu: str, cl: str},
                              chunksize=CHUNK):
            rows += len(ch)
            tu = pd.to_datetime(ch[cu], format=fmt, errors="coerce")
            tl = pd.to_datetime(ch[cl], format=fmt, errors="coerce")
            bad_ts += int(tu.isna().sum())
            vals = ch[irr].apply(pd.to_numeric, errors="coerce")
            s_mask = (vals >= 9000) | (vals <= -999)                       # logger fill codes
            sent += int(s_mask.to_numpy().sum())
            g = vals.mask(s_mask).mean(axis=1)
            tc = ch[tmp].apply(pd.to_numeric, errors="coerce").mean(axis=1) if tmp else pd.Series(np.nan, index=ch.index)
            off = ((tl - tu).dt.total_seconds() / 3600).round(4).value_counts()
            for k, v in off.items():
                offsets[k] = offsets.get(k, 0) + int(v)
            d = tu.diff().dt.total_seconds()
            if last_t is not None and len(tu):
                d.iloc[0] = (tu.iloc[0] - last_t).total_seconds()
            nonmono += int((d <= 0).sum()); gaps += int((d > 1.5 * np.nanmedian(d)).sum())
            last_t = tu.iloc[-1] if len(tu) else last_t
            for tt, acc, extra in ((tl, acc_local, tc), (tu + pd.Timedelta(hours=9.5), acc_acst, None)):
                hb = tt.dt.floor("h")
                f = pd.DataFrame({"ts": hb, "g": g, "o": (tt - hb).dt.total_seconds()})
                if extra is not None:
                    f["tc"] = extra
                agg = f.groupby("ts").agg(g_sum=("g", "sum"), g_n=("g", "count"), o_sum=("o", "sum"), n=("o", "count"),
                                          **({"tc_sum": ("tc", "sum"), "tc_n": ("tc", "count")} if extra is not None else {}))
                acc.append(agg)
            m = (tu >= JULY[0]) & (tu < JULY[1])
            if m.any():
                july.append(pd.DataFrame({"raw_utc": ch.loc[m, cu].values, "raw_local": ch.loc[m, cl].values,
                                          "ghi": g[m].values}))
        facts.append({"file": os.path.basename(path), "rows": rows, "timestamp_format": fmt,
                      "unparseable_ts": bad_ts, "non_monotonic_or_duplicate": nonmono, "gaps_>1.5x_step": gaps,
                      "sentinels_masked": sent, "local_minus_utc_h": offsets,
                      "seconds": round(time.time() - t0)})
        print(f"  {os.path.basename(path)}: {rows:,} rows in {time.time()-t0:.0f}s")
    def fin(acc):
        a = pd.concat(acc).groupby(level=0).sum()
        out = pd.DataFrame({"ts": a.index, "ghi_raw": a.g_sum / a.g_n.replace(0, np.nan),
                            "n_obs": a.g_n.astype(int), "ts_off_min": a.o_sum / a.n / 60.0})
        if "tc_sum" in a:
            out["unit_temp_c"] = (a.tc_sum / a.tc_n.replace(0, np.nan)).values
        return out.reset_index(drop=True)
    return fin(acc_local), fin(acc_acst), (pd.concat(july, ignore_index=True) if july else pd.DataFrame()), facts

if all(os.path.exists(p) for p in (P_LOCAL, P_ACST, P_FACTS)):
    H_LOCAL, H_ACST = pd.read_parquet(P_LOCAL), pd.read_parquet(P_ACST)
    JULY_1S = pd.read_parquet(P_JULY) if os.path.exists(P_JULY) else pd.DataFrame()
    INGEST_FACTS = json.load(open(P_FACTS))
    print("🔬 loaded cached hourly aggregates (delete files in CACHE to recompute)")
else:
    print("🔬 streaming Darwin 1-s files (first run only) ...")
    H_LOCAL, H_ACST, JULY_1S, INGEST_FACTS = ingest_darwin_1s(FILES["darwin_1s"])
    H_LOCAL.to_parquet(P_LOCAL); H_ACST.to_parquet(P_ACST)
    if len(JULY_1S): JULY_1S.to_parquet(P_JULY)
    json.dump(INGEST_FACTS, open(P_FACTS, "w"), default=str)
print(f"hourly bins — local basis: {len(H_LOCAL):,}   ACST basis: {len(H_ACST):,}   July-2022 1-s rows kept: {len(JULY_1S):,}")
assert H_LOCAL.n_obs.sum() == H_ACST.n_obs.sum(), "both bases must aggregate the same samples"
print("✔ both time bases aggregate exactly the same number of valid samples:", f"{int(H_LOCAL.n_obs.sum()):,}")

## 5.2 Data validation (🔬 RECALCULATED)

**What.** Dataset dimensions, time range, sampling interval, unparseable or duplicate timestamps, gaps, logger fill values, and the offset between the two timestamp columns, all on every raw row.
**Why.** v5 performed no row-level validation. Its audit was file-level. These are the checks the later versions added, applied here to the same data.

In [ ]:
facts_df = pd.DataFrame(INGEST_FACTS)
facts_df["local_minus_utc_h"] = facts_df["local_minus_utc_h"].map(lambda d: {float(k): v for k, v in d.items()})
display(facts_df)
off_all = {}
for d in facts_df.local_minus_utc_h:
    for k, v in d.items():
        off_all[k] = off_all.get(k, 0) + v
OFFSET_TABLE = pd.Series(off_all, name="rows").rename_axis("Timestamp_Local − Timestamp_UTC (h)").to_frame()
OFFSET_TABLE["share_%"] = 100 * OFFSET_TABLE.rows / OFFSET_TABLE.rows.sum()
display(OFFSET_TABLE)
print("Darwin (Northern Territory) legal time: ACST = UTC+9.5 h all year (no daylight saving).")
for name, H in (("local basis", H_LOCAL), ("ACST basis", H_ACST)):
    full = pd.date_range(H.ts.min(), H.ts.max(), freq="h")
    print(f"{name:<12}: {H.ts.min()} → {H.ts.max()} | hours with data {len(H):,} of {len(full):,} "
          f"({100*len(H)/len(full):.1f}%) | median samples/hour {H.n_obs.median():,.0f}")

## 5.3 Timestamp handling in v5 (quoted from the v5 notebook)

v5 `load_darwin()` (cell 17):

```python
h = stream_to_hourly(p, "timestamp_local", "%d/%m/%Y %H:%M:%S", irr + tmp, ...)
```

v5 `attach_solar()` (cell 9) then evaluates the sun **at the hourly label**:

```python
cosz, toa = solar_geometry(pd.DatetimeIndex(df["ts"]), m["lat"], m["lon"], m["tz"])   # tz = 9.5 (ACST)
```

**Two implicit assumptions**, both tested later:

1. `Timestamp_Local` is ACST.
2. An hourly mean represents the instant HH:00.

The shared, version-independent physics below is copied verbatim from the pipeline (NOAA solar position with Spencer terms, and the Haurwitz clear-sky model).

In [ ]:
def solar_geometry(ts_local, lat, lon, tz):
    utc = pd.DatetimeIndex(ts_local) - pd.Timedelta(hours=tz)
    doy = np.asarray(utc.dayofyear, float)
    hod = np.asarray(utc.hour, float) + np.asarray(utc.minute, float) / 60 + np.asarray(utc.second, float) / 3600
    g = 2 * np.pi * (doy - 1 + (hod - 12) / 24) / 365
    eqt = 229.18 * (0.000075 + 0.001868*np.cos(g) - 0.032077*np.sin(g) - 0.014615*np.cos(2*g) - 0.040849*np.sin(2*g))
    decl = (0.006918 - 0.399912*np.cos(g) + 0.070257*np.sin(g) - 0.006758*np.cos(2*g)
            + 0.000907*np.sin(2*g) - 0.002697*np.cos(3*g) + 0.001480*np.sin(3*g))
    ha = np.radians((hod * 60 + eqt + 4 * lon) / 4 - 180)
    la = np.radians(lat)
    return np.clip(np.sin(la)*np.sin(decl) + np.cos(la)*np.cos(decl)*np.cos(ha), -1, 1)

def clearsky_haurwitz(cosz):
    c = np.where(cosz > 0, cosz, np.nan)
    with np.errstate(all="ignore"):
        return np.nan_to_num(1098.0 * c * np.exp(-0.059 / c))

## 5.4 Data cleaning: QC, clear-sky index and gap-aware interpolation

**What.** `build_version()` reproduces the pipeline's `qc_and_kt()` and `impute()`:

* hard negatives and values above 1,400 W/m² → missing;
* GHI above 1.3 × clear-sky → flagged **non-physical** and removed;
* night → structural zero;
* k_t = GHI / GHI_cs, capped at 1.3;
* daylight gaps ≤ 3 h interpolated in k_t space; longer gaps left missing.

The **only** arguments that change between versions are the *hourly bins* (local or ACST basis) and the *instant used for solar geometry* (bin start or mean sample time). Everything else is identical, so any difference between `df_v5`, `df_v6` and `df_v7` is caused by timing alone.

In [ ]:
def build_version(H, site, geometry, label):
    # H: hourly aggregates; geometry: "bin_start" (v5) or "sample_centre" (v6, v7)
    d = H.set_index("ts").reindex(pd.date_range(H.ts.min(), H.ts.max(), freq="h")).rename_axis("ts").reset_index()
    off = d["ts_off_min"].fillna(d["ts_off_min"].median() if d["ts_off_min"].notna().any() else 30.0)
    gt = pd.DatetimeIndex(d["ts"]) + (pd.to_timedelta(off, unit="m") if geometry == "sample_centre" else pd.Timedelta(0))
    cosz = solar_geometry(gt, site["lat"], site["lon"], site["tz"])
    d["cos_zenith"], d["solar_elev_deg"] = cosz, 90 - np.degrees(np.arccos(cosz))
    d["ghi_cs"] = clearsky_haurwitz(cosz)
    d["is_daylight"] = d["ghi_cs"] > DAYLIGHT_CS_MIN
    g = d["ghi_raw"]
    d["qc_raw_missing"] = g.isna()
    g = g.where(g >= NEG_TOL).clip(lower=0).where(lambda x: x <= GHI_MAX)
    d["qc_nonphysical"] = ((g > KT_MAX * d["ghi_cs"]) & d["is_daylight"]).fillna(False)
    g = g.where(~d["qc_nonphysical"]).where(d["is_daylight"], 0.0)
    with np.errstate(all="ignore"):
        kt = np.where(d["is_daylight"], g / d["ghi_cs"].replace(0, np.nan), 0.0)
    kt = pd.Series(np.clip(kt, 0, KT_MAX)).where(g.notna().to_numpy())
    miss = kt.isna() & d["is_daylight"]
    run_id = (miss != miss.shift()).cumsum()
    run_len = miss.groupby(run_id).transform("sum")
    short = miss & (run_len <= MAX_INTERP_GAP_H)
    kt_i = kt.interpolate(limit=MAX_INTERP_GAP_H, limit_area="inside")
    d["kt"] = np.where(short, kt_i, kt)
    d["kt_imputed"] = short & pd.Series(d["kt"]).notna()
    d["kt_unrecoverable"] = miss & ~short
    d["ghi_wm2"] = np.where(pd.Series(d["kt"]).notna(), d["kt"] * d["ghi_cs"], np.nan)
    d["version"] = label
    return d

def engineer(d, site):
    # identical to pipeline engineer() — every rolling statistic is shifted so it uses only the past
    d = d.copy(); ts = d["ts"]
    d["hour_sin"], d["hour_cos"] = np.sin(2*np.pi*ts.dt.hour/24), np.cos(2*np.pi*ts.dt.hour/24)
    d["doy_sin"], d["doy_cos"] = np.sin(2*np.pi*ts.dt.dayofyear/365.25), np.cos(2*np.pi*ts.dt.dayofyear/365.25)
    d["is_wet"] = ts.dt.month.isin(site["wet_months"]).astype(float)
    for L in (1, 2, 3, 24):
        d[f"kt_lag{L}"] = d["kt"].shift(L)
    for W in (3, 24):
        r = d["kt"].rolling(W, min_periods=max(2, W // 3))
        d[f"kt_roll{W}_mean"], d[f"kt_roll{W}_std"] = r.mean().shift(1), r.std().shift(1)
    d["ghi_cs_norm"] = d["ghi_cs"] / 1100.0
    d["dkt"] = d["kt"].diff()
    d["is_ramp_raw"] = (d["dkt"].abs() > RAMP_KT) & d["is_daylight"]                       # v5 definition
    d["is_ramp_kt"] = d["is_ramp_raw"] & (d["solar_elev_deg"] > ELEV_FLOOR)                # v6+ definition
    return d

FEATURES = ["kt", "kt_lag1", "kt_lag2", "kt_lag3", "kt_lag24", "kt_roll3_mean", "kt_roll3_std",
            "kt_roll24_mean", "kt_roll24_std", "cos_zenith", "ghi_cs_norm",
            "hour_sin", "hour_cos", "doy_sin", "doy_cos", "is_wet"]          # final 16-feature vector

In [ ]:
# ---------- v5: ORIGINAL implementation (Local-column bins, sun at bin start) ----------
df_v5 = engineer(build_version(H_LOCAL, DARWIN, "bin_start", "v5"), DARWIN)
df_v5["temp_c"] = H_LOCAL.set_index("ts")["unit_temp_c"].reindex(df_v5.ts).to_numpy()   # v5: reference-CELL temperature
print(f"df_v5: {len(df_v5):,} hourly rows, {int(df_v5.is_daylight.sum()):,} daylight hours")
display(df_v5[["ts", "ghi_raw", "n_obs", "ghi_cs", "kt", "qc_nonphysical", "kt_imputed", "temp_c"]]
        .loc[df_v5.is_daylight].head(8))

In [ ]:
qc_summary = lambda d: pd.Series({
    "hourly rows": len(d), "daylight hours": int(d.is_daylight.sum()),
    "raw missing (daylight) %": 100 * (d.qc_raw_missing & d.is_daylight).sum() / d.is_daylight.sum(),
    "non-physical removed (h)": int(d.qc_nonphysical.sum()),
    "k_t interpolated (h)": int(d.kt_imputed.sum()),
    "k_t unrecoverable (h)": int(d.kt_unrecoverable.sum()),
    "valid daylight k_t (h)": int((pd.Series(d.kt).notna() & d.is_daylight).sum())})
print("🔬 v5 QC LEDGER (Darwin, 2021-2023)")
display(qc_summary(df_v5).round(2).to_frame("v5"))

## 5.5 v5 exploratory analysis (🔬 recalculated for Darwin; 📓 v5's all-site figures reproduced)

**Clear-sky validation (v5 Figure 8).** v5 plotted measured against modelled clear-sky GHI. With correct timing, clear hours lie along one line. Two separate branches mean the model and the measurement disagree in the morning and afternoon in opposite directions, which is the signature of a timing error. v6's code comments explicitly cite this v5 figure as the evidence that prompted the v6 change.

In [ ]:
def clear_hours(d):
    daily = d[d.is_daylight].groupby(d.ts.dt.floor("D")).agg(g=("ghi_wm2", "sum"), cs=("ghi_cs", "sum"),
                                                             n=("ghi_wm2", "count"))
    clear_days = daily[(daily.n >= 8) & (daily.g / daily.cs >= 0.75)].index
    return d[d.ts.dt.floor("D").isin(clear_days) & d.is_daylight & (d.ghi_cs > 50)]

c5 = clear_hours(df_v5)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
am = c5.ts.dt.hour < 12
ax[0].scatter(c5.ghi_cs[am], c5.ghi_raw[am], s=6, alpha=.5, label="morning hours")
ax[0].scatter(c5.ghi_cs[~am], c5.ghi_raw[~am], s=6, alpha=.5, label="afternoon hours")
ax[0].plot([0, 1100], [0, 1100], "k--", lw=1); ax[0].set_xlabel("modelled clear-sky GHI (W/m²)"); ax[0].set_ylabel("measured GHI (W/m²)")
ax[0].set_title("v5 clear hours: two branches = timing error", fontsize=9.5, loc="left"); ax[0].legend(fontsize=8)
ax[1].hist(pd.Series(df_v5.kt)[df_v5.is_daylight].dropna(), bins=60, color="#c1440e")
ax[1].set_xlabel("daylight k_t (v5)"); ax[1].set_title("v5 k_t distribution (note mass at the 1.3 cap)", fontsize=9.5, loc="left")
plt.tight_layout(); plt.show()
print(f"🔬 clear days found: {c5.ts.dt.floor('D').nunique()};  v5 clear-hour measured/model ratio: "
      f"morning {np.median(c5.ghi_raw[am]/c5.ghi_cs[am]):.2f}, afternoon {np.median(c5.ghi_raw[~am]/c5.ghi_cs[~am]):.2f}")

In [ ]:
for f in ("fig01_availability", "fig02_diurnal", "fig03_kt_dist", "fig08_clearsky"):
    show_reproduced("v5", f, width=900)

## 5.6 v5 feature engineering

v5 built 19 candidate inputs and screened out any feature that was more than 50% imputed or constant at the primary site:

* 9 **k_t-history** features: level, lags 1/2/3/24 h, and 3 h and 24 h rolling mean/SD, all shifted to use only the past;
* **solar geometry**: cos zenith, clear-sky level;
* **calendar**: hour and day-of-year sine/cosine;
* **meteorology**: temperature, humidity, wind;
* **season**: `is_wet`.

At Darwin, v5's `temp_c` was the **reference-cell body temperature**, not air temperature, and humidity and wind did not exist. The screen below reproduces the outcome.

In [ ]:
def screen(d, candidates):
    rows = []
    for f in candidates:
        s = pd.to_numeric(d[f], errors="coerce") if f in d else pd.Series(np.nan, index=d.index)
        imp = float(s.isna().mean()); sd = float(s.std(skipna=True) or 0)
        rows.append({"feature": f, "missing_frac": round(imp, 3), "std": round(sd, 4),
                     "decision": "DROP" if (imp > 0.5 or sd < 1e-6) else "keep"})
    return pd.DataFrame(rows)
v5_candidates = FEATURES[:11] + ["temp_c", "rh_pct", "wind_ms"] + FEATURES[11:]
SCREEN_V5 = screen(df_v5, v5_candidates)
display(SCREEN_V5)
print("v5 modelling vector:", SCREEN_V5[SCREEN_V5.decision == "keep"].feature.tolist())

## 5.7 Modelling and forecast preparation

**What.**

* Supervised windows: 24 h of features → k_t at t+1, daylight targets only, any window with a missing value discarded.
* Split: chronological 70/15/15, never shuffled; scaler fitted on training rows only.
* References: persistence, climatology (month × hour median, training rows only) and seasonal naive.
* Models: GRU and QR-GRU with v5's configuration (64 units, 24 h, lr 1e-3, early stopping on validation).

These functions are reused unchanged for every version, so the controlled experiments in §6–§8 differ **only** in the data version.

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
tf.keras.utils.set_random_seed(SEED)

def build_supervised(d, feats=FEATURES, target="kt", lookback=LOOKBACK, horizon=HORIZON):
    X_all = d[feats].to_numpy(dtype="float32")
    y_all = pd.to_numeric(d[target], errors="coerce").to_numpy(dtype="float32")
    n = len(d) - lookback - horizon + 1
    if n <= 0: return None
    W = sliding_window_view(X_all, (lookback, len(feats)))[:n, 0]
    tgt = np.arange(n) + lookback - 1 + horizon
    ok = np.isfinite(W).all(axis=(1, 2)) & np.isfinite(y_all[tgt]) & d["is_daylight"].to_numpy()[tgt]
    meta = d.iloc[tgt[ok]][["ts", "ghi_cs", "is_wet", "is_ramp_kt", "kt_imputed"]].reset_index(drop=True)
    return W[ok], y_all[tgt[ok]], meta

def split_chrono(X, y, meta, fracs=(0.70, 0.15)):
    n = len(y); edges = np.cumsum([0] + [int(n * f) for f in fracs] + [n])
    edges[-1] = n
    names = ["train", "val", "cal", "test"][:len(fracs)] + ["test"]
    names = ["train", "val", "test"] if len(fracs) == 2 else ["train", "val", "cal", "test"]
    return {k: (X[a:b], y[a:b], meta.iloc[a:b].reset_index(drop=True)) for k, a, b in zip(names, edges[:-1], edges[1:])}

class Standardiser:
    def fit(self, X):
        f = X.reshape(-1, X.shape[-1]); self.m, self.s = f.mean(0), f.std(0) + 1e-6; return self
    def __call__(self, X): return (X - self.m) / self.s

def pinball_loss(q):
    q = tf.constant(q, tf.float32)
    def loss(y, p):
        e = tf.cast(y, tf.float32)[:, None] - p
        return tf.reduce_mean(tf.maximum(q * e, (q - 1) * e))
    return loss

def gaussian_nll(y, p):
    y = tf.reshape(tf.cast(y, tf.float32), [-1]); mu, lv = p[:, 0], tf.clip_by_value(p[:, 1], -12., 6.)
    return tf.reduce_mean(0.5 * (lv + tf.square(y - mu) * tf.exp(-lv)))

def make_model(kind, nf, lookback=LOOKBACK, units=UNITS):
    inp = keras.Input((lookback, nf))
    if kind == "GRU":
        x = layers.Dense(32, activation="relu")(layers.GRU(units)(inp)); out, loss = layers.Dense(1)(x), "mse"
    elif kind == "QR-GRU":
        x = layers.Dense(32, activation="relu")(layers.GRU(units)(inp)); out, loss = layers.Dense(len(QUANTILES))(x), pinball_loss(QUANTILES)
    elif kind in ("MC-Dropout LSTM", "HMC-LSTM"):
        x = layers.Dropout(0.2)(layers.LSTM(units, dropout=0.2)(inp), training=True)
        x = layers.Dropout(0.2)(layers.Dense(32, activation="relu")(x), training=True)
        out, loss = (layers.Dense(1)(x), "mse") if kind == "MC-Dropout LSTM" else (layers.Dense(2)(x), gaussian_nll)
    m = keras.Model(inp, out); m.compile(keras.optimizers.Adam(LR), loss=loss); return m

def fit_predict(kind, sp, seed=SEED, passes=50):
    tf.keras.utils.set_random_seed(seed)
    sc = Standardiser().fit(sp["train"][0])
    m = make_model(kind, sp["train"][0].shape[-1])
    h = m.fit(sc(sp["train"][0]), sp["train"][1], validation_data=(sc(sp["val"][0]), sp["val"][1]), epochs=EPOCHS,
              batch_size=BATCH, verbose=0, callbacks=[keras.callbacks.EarlyStopping(patience=PATIENCE, restore_best_weights=True)])
    def pred(X):
        if kind in ("MC-Dropout LSTM", "HMC-LSTM"):
            outs = [m(sc(X), training=True).numpy() for _ in range(passes)]
            if kind == "MC-Dropout LSTM":
                S = np.stack([o.ravel() for o in outs], 1)
            else:
                rng = np.random.default_rng(seed)
                mu = np.stack([o[:, 0] for o in outs], 1); sd = np.exp(0.5 * np.clip(np.stack([o[:, 1] for o in outs], 1), -12, 6))
                S = mu + sd * rng.standard_normal(mu.shape)
            return S.mean(1), np.quantile(S, QUANTILES, axis=1).T
        if kind == "QR-GRU":
            Q = np.sort(m.predict(sc(X), verbose=0), 1); return Q[:, QUANTILES.index(0.5)], Q
        return m.predict(sc(X), verbose=0).ravel(), None
    out = {k: pred(sp[k][0]) for k in sp if k != "train"}
    tf.keras.backend.clear_session()
    return out, len(h.history["loss"])

def climatology(sp):
    _, ytr, mtr = sp["train"]
    tab = pd.DataFrame({"kt": ytr, "m": mtr.ts.dt.month, "h": mtr.ts.dt.hour}).groupby(["m", "h"]).kt.median()
    g = float(np.median(ytr))
    return lambda meta: tab.reindex(pd.MultiIndex.from_arrays([meta.ts.dt.month, meta.ts.dt.hour])).fillna(g).to_numpy()

def score(p_kt, y_kt, meta, Q=None, delta=0.0):
    cs = meta.ghi_cs.to_numpy(); yt, yh = y_kt * cs, np.clip(p_kt * cs, 0, None)
    r = {"n": len(yt), "RMSE": float(np.sqrt(np.mean((yh - yt) ** 2))), "MAE": float(np.mean(np.abs(yh - yt)))}
    if Q is not None:
        lo, hi = np.clip((Q[:, I_LO] - delta) * cs, 0, None), np.clip((Q[:, I_HI] + delta) * cs, 0, None)
        r["PICP80"], r["MIW"] = float(((yt >= lo) & (yt <= hi)).mean()), float((hi - lo).mean())
        Qw = np.clip(Q * cs[:, None], 0, None)
        pl = np.stack([np.maximum(t * (yt - Qw[:, j]), (t - 1) * (yt - Qw[:, j])) for j, t in enumerate(QUANTILES)], 1)
        r["CRPS_q"] = float(np.mean(2 * np.trapezoid(pl, QUANTILES, axis=1)) if hasattr(np, "trapezoid") else np.mean(2 * np.trapz(pl, QUANTILES, axis=1)))
    return r

def cqr_delta(Q, y, alpha=0.2):
    E = np.maximum(Q[:, I_LO] - y, y - Q[:, I_HI]); k = min(len(E), int(np.ceil((len(E) + 1) * (1 - alpha))))
    return float(np.sort(E)[k - 1])
print("modelling functions ready; TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 5.8 v5 results (📓 REPRODUCED from the executed v5 notebook)

These are v5's own full-scale outputs: four architectures, seed 42, one test block of 535 windows.

In [ ]:
print(executed_text("v5", "MAIN RESULTS", 3000) if _nb("v5") else "executed v5 notebook not found")
for f in ("fig11_forecast", "fig12_calibration", "fig16_energy_regimes", "fig23_leaderboard"):
    show_reproduced("v5", f, width=950)

## 5.9 What v5 taught us, and its problems

**What v5 established.** A complete, reproducible pipeline covering streaming ingestion, QC, EDA, features, four architectures, proper scores, calibration, ablation, proxy, transfer and decision support. Its headline was that QR-GRU beat persistence by 21.8%.

**Problems identified in v5**, each examined in the section shown:

| # | Problem | Found by | Section |
|---|---|---|---|
| P1 | Sun evaluated at bin start; the hourly mean represents ~HH:30 | v5 Figure 8 (two branches) | §6 |
| P2 | `Timestamp_Local` assumed to be ACST | not detected in v5 or v6; found by the v7 clear-day gate | §7, §11 |
| P3 | Single test block, 3 seeds, DM test without autocorrelation correction | v6 seed spread ≈ architecture spread | §7 |
| P4 | Seasonal ablation removed `is_wet` but kept `doy_sin/cos` | inspection of the feature vector | §7, §12 |
| P5 | Ramp definition fires at dawn/dusk (tiny denominator) | 5A ramp analysis | §9 |
| P6 | Darwin `temp_c` = reference-cell temperature | unit registry audit | §10 |
| P7 | Conformal calibration on the early-stopping split | §12 review | §12 |

---
# 6. Version 6 — first major enhancement

**Retained from v5:** every analytical part, the same Darwin data, the same split, models, seeds and metrics (v6 change log).
**Added:** a raw file inventory (1.0), row-level data validation with a diurnal-lag check (3A), input fingerprinting of checkpoints, extended sites, and a leakage diagnostic.

## 6.1 Problem identified (P1)

An hourly mean of 1-second samples describes the **middle** of its hour, but v5 computed the sun at the **start**. The clear-sky reference was therefore about 30 minutes early relative to the measurement it divides.

## 6.2 Evidence

In [ ]:
print("Mean within-bin sample offset (minutes after the HH:00 label), local basis:")
print(H_LOCAL.ts_off_min.describe()[["mean", "50%", "min", "max"]].round(2).to_string())

**Observation.** The average sample in every hourly bin sits about 30 minutes after the label. Evaluating the sun at HH:00 is therefore systematically early by that amount.

## 6.3 Modification: v6 evaluates the sun at each bin's mean sample time (`geometry_times`)

In [ ]:
# ---------- v6: CORRECTED geometry instant (still Local-column bins) ----------
df_v6 = engineer(build_version(H_LOCAL, DARWIN, "sample_centre", "v6"), DARWIN)
df_v6["temp_c"] = df_v5["temp_c"].values
same_ghi = np.allclose(df_v5.ghi_raw, df_v6.ghi_raw, equal_nan=True)
print(f"df_v6: {len(df_v6):,} rows.  Hourly GHI values identical to v5: {same_ghi}  (v6 changed geometry only)")
assert same_ghi

## 6.4 Before vs after (🔬 RECALCULATED, same data)

In [ ]:
def kt_tilt(d):
    c = clear_hours(d); k = pd.Series(c.kt.values, index=c.ts.dt.hour)
    return pd.Series({"clear-day k_t morning (8-11h)": k[k.index.isin(range(8, 12))].median(),
                      "clear-day k_t afternoon (13-16h)": k[k.index.isin(range(13, 17))].median()})
cmp56 = pd.concat({"v5": pd.concat([qc_summary(df_v5), kt_tilt(df_v5)]),
                   "v6": pd.concat([qc_summary(df_v6), kt_tilt(df_v6)])}, axis=1).round(3)
cmp56.loc["morning − afternoon"] = cmp56.loc["clear-day k_t morning (8-11h)"] - cmp56.loc["clear-day k_t afternoon (13-16h)"]
display(cmp56)

**Interpretation.**

* A correctly timed clear day has a flat k_t, so morning ≈ afternoon.
* v5's large morning excess means its target was biased high in the morning and low in the afternoon. It also pushed morning hours over the 1.3 ceiling, where QC *removed* them: more "non-physical" hours in v5.
* v6 reduces both effects.

The same measurements and the same code produce different modelling data. **Demonstrated.**

### 6.4.1 Controlled experiment A: does the timing change alone move the forecasting results? (🔬 RECALCULATED)

**Design.** Identical Darwin measurements, identical 70/15/15 split logic, identical models and seeds (42 and 7). Only the data version differs. The metrics are in W/m², so v5 and v6 are scored on **identical GHI values**.
**Cost.** 8 small fits, about 3–5 min on a GPU.

In [ ]:
EXP_A = {}
def run_version(name, d, seeds=(42, 7)):
    X, y, meta = build_supervised(d)
    sp = split_chrono(X, y, meta)
    Xte, yte, mte = sp["test"]
    rows = [{"version": name, "model": "Persistence", "seed": "-", **score(Xte[:, -1, 0], yte, mte)},
            {"version": name, "model": "Climatology", "seed": "-", **score(climatology(sp)(mte), yte, mte)}]
    for kind in ("GRU", "QR-GRU"):
        for s in seeds:
            out, ep = fit_predict(kind, sp, seed=s)
            p, Q = out["test"]
            rows.append({"version": name, "model": kind, "seed": s, "epochs": ep, **score(p, yte, mte, Q)})
    EXP_A[name] = sp
    return pd.DataFrame(rows)

t0 = time.time()
RES_A = pd.concat([run_version("v5", df_v5), run_version("v6", df_v6)], ignore_index=True)
pers = RES_A[RES_A.model == "Persistence"].set_index("version").RMSE
RES_A["skill_%"] = 100 * (1 - RES_A.RMSE / RES_A.version.map(pers))
print(f"experiment A (v5 vs v6) in {time.time()-t0:.0f}s")
display(RES_A.round(3))

**Reading the result.** Compare persistence RMSE and model skill between v5 and v6. The measurements, split and models are identical, so any difference is caused by the ~30-minute change in where the sun is evaluated. The full-scale v5 → v6 runs recorded the same kind of shift: persistence 146.8 → 129.3 W/m², QR-GRU skill 21.8% → 7.0% (📌 §0.2).

**Demonstrated:** results built on k_t are sensitive to timing.

## 6.5 Research implication

The v5 headline (+21.8%) was partly a product of a timing artefact. The artefact made persistence look worse, because persistence carries forward a k_t that is biased in a direction that changes through the day.

### What v6 resolved
* P1: the within-hour geometry instant.
* v6 also added row-level validation and a diurnal-lag diagnostic.

### What remained unresolved, and what v6 made worse

| Issue | Status after v6 |
|---|---|
| P2 `Timestamp_Local` reference | **Still present.** v6's lag check measured −31 min at Darwin but passed it under a ±45 min tolerance |
| P3 single split, DM without HAC | Present. v6's own seed spread (SD ≈ 30 W/m²) ≈ the architecture spread (33.7 W/m²) |
| P4 seasonal ablation confounded | Present, and its sign flipped between v5 and v6 |
| **New:** Singapore arm | **Regressed.** A 1-month Open-Meteo file replaced 14 years of NASA POWER (8,572 → 51 test windows), and the change log misreported the source |
| **New:** Darwin met files | Present on Drive but left unclassified; Part 15 then stated that "Darwin has no meteorology" |
| Narrative | Several markdown cells quoted v5 numbers; Part 14 and 14B gave contradictory values |

In [ ]:
show_reproduced("v6", "figV2_timing", width=950)
print(executed_text("v6", "DATA VALIDATION", 2500) if _nb("v6") else "")

---
# 7. Version 7 — further research development

**Inherited from v6:** the full pipeline, the v6 geometry instant, validation and extended sites.
**Changed:** the timestamp reference, the evaluation protocol, the statistical tests, the seasonal-ablation design, calibration and tuning, the Darwin record, the Singapore source, and the use of Darwin meteorology. Each change is examined below with **before → problem → modification → after**.

## 7.1 Problem P2: `Timestamp_Local` is not Darwin time

### Evidence (🔬 RECALCULATED)
§5.2 already measured the offset on every raw row. Here is the consequence at the hourly level; the physical proof is in §11.

In [ ]:
print("Timestamp_Local − Timestamp_UTC over all ingested rows:")
display(OFFSET_TABLE)
print("If Timestamp_Local were ACST, this would be 9.5 h on every row.")

In [ ]:
# ---------- v7: CORRECTED implementation (UTC column + 9:30 = ACST bins, sun at bin centre) ----------
df_v7 = engineer(build_version(H_ACST, DARWIN, "sample_centre", "v7"), DARWIN)
df_v71 = df_v7            # v7.1 changed ONLY the proxy test (§8); the Darwin data are identical — stated, not assumed
print(f"df_v7: {len(df_v7):,} rows | samples aggregated equal to v5/v6: {H_ACST.n_obs.sum() == H_LOCAL.n_obs.sum()}")
cmp_all = pd.concat({v: pd.concat([qc_summary(d), kt_tilt(d)]) for v, d in (("v5", df_v5), ("v6", df_v6), ("v7", df_v7))}, axis=1)
cmp_all.loc["morning − afternoon"] = cmp_all.loc["clear-day k_t morning (8-11h)"] - cmp_all.loc["clear-day k_t afternoon (13-16h)"]
display(cmp_all.round(3))

**After.** Only v7 gives a flat clear-day k_t. The number of hours removed as "non-physical" falls, because morning hours are no longer pushed over the 1.3 ceiling.

**Hourly values changed as well as labels.** A local-column bin HH:00 contains ACST HH:30–HH+1:30, so v5/v6 and v7 hourly GHI are **different numbers**: same raw samples, different grouping. **Demonstrated** (§11).

### Controlled experiment A, continued: the v7 timing on the same models (🔬 RECALCULATED)

In [ ]:
RES_A = pd.concat([RES_A, run_version("v7", df_v7)], ignore_index=True)
pers = RES_A[RES_A.model == "Persistence"].groupby("version").RMSE.first()
RES_A["skill_%"] = 100 * (1 - RES_A.RMSE / RES_A.version.map(pers))
summ_A = RES_A.groupby(["version", "model"]).agg(RMSE=("RMSE", "mean"), skill=("skill_%", "mean"),
                                                   PICP80=("PICP80", "mean"), n=("n", "first")).round(3)
display(summ_A)
print("Note: v7 is scored on its own (correctly binned) hourly GHI, so its RMSE is comparable in skill, "
      "not in absolute W/m², to v5/v6.")

## 7.2 Problem P3: one test block cannot rank architectures, and the old DM test was anti-conservative

**Before (v5/v6).** One 535-window test block, 3 seeds. The Diebold–Mariano test at h = 1 used **no** autocorrelation correction.
**Evidence.** In v6 the seed SD (~30 W/m²) was about the size of the whole architecture spread (📌 recorded). Below, the v7 data are evaluated over four rolling origins, which shows how much the answer depends on which period is tested. The same errors are then passed to the naive and HAC-corrected DM tests.

In [ ]:
X7, y7, m7 = build_supervised(df_v7)
n7 = len(y7); start = int(n7 * 0.6); edges = np.linspace(start, n7, 5).astype(int)
fold_rows, pooled = [], {"GRU": [], "QR-GRU": [], "Persistence": [], "fold": []}
for k in range(4):
    t0_, t1_ = edges[k], edges[k + 1]
    v_n = int(t0_ * 0.125 / 0.85); c_n = int(t0_ * 0.125 / 0.85)
    a_ = t0_ - v_n - c_n; b_ = t0_ - c_n
    sp = {"train": (X7[:a_], y7[:a_], m7.iloc[:a_]), "val": (X7[a_:b_], y7[a_:b_], m7.iloc[a_:b_].reset_index(drop=True)),
          "cal": (X7[b_:t0_], y7[b_:t0_], m7.iloc[b_:t0_].reset_index(drop=True)),
          "test": (X7[t0_:t1_], y7[t0_:t1_], m7.iloc[t0_:t1_].reset_index(drop=True))}
    Xte, yte, mte = sp["test"]; cs = mte.ghi_cs.to_numpy()
    rp = score(Xte[:, -1, 0], yte, mte)["RMSE"]
    pooled["Persistence"].append((Xte[:, -1, 0] - yte) * cs); pooled["fold"].append(np.full(len(yte), k + 1))
    for kind in ("GRU", "QR-GRU"):
        out, _ = fit_predict(kind, sp, seed=42)
        p, Q = out["test"]
        pooled[kind].append((np.clip(p * cs, 0, None) - yte * cs))
        fold_rows.append({"fold": k + 1, "test_from": mte.ts.min(), "test_to": mte.ts.max(), "wet_%": 100 * mte.is_wet.mean(),
                          "model": kind, "RMSE": score(p, yte, mte)["RMSE"], "persistence_RMSE": rp})
FOLDS7 = pd.DataFrame(fold_rows); FOLDS7["skill_%"] = 100 * (1 - FOLDS7.RMSE / FOLDS7.persistence_RMSE)
display(FOLDS7.round(2))
print("🔬 spread of GRU skill across periods: "
      f"{FOLDS7[FOLDS7.model=='GRU']['skill_%'].min():.1f}% to {FOLDS7[FOLDS7.model=='GRU']['skill_%'].max():.1f}%")

In [ ]:
from scipy import stats
def dm_test(e1, e2, lag):
    d = e1 ** 2 - e2 ** 2; n = len(d); dc = d - d.mean(); s = (dc ** 2).mean()
    for k in range(1, lag + 1):
        s += 2 * (1 - k / (lag + 1)) * (dc[k:] * dc[:-k]).mean()
    stat = d.mean() / np.sqrt(s / n)
    return stat, 2 * (1 - stats.t.cdf(abs(stat), n - 1))
eG, eQ = np.concatenate(pooled["GRU"]), np.concatenate(pooled["QR-GRU"])
d = eG ** 2 - eQ ** 2
ac1 = pd.Series(d).autocorr(1)
L_nw = int(np.floor(4 * (len(d) / 100) ** (2 / 9)))
for name, lag in (("v5/v6 DM (no autocorrelation correction, lag 0)", 0), (f"v7 DM (Newey–West, lag {L_nw})", L_nw)):
    st, p = dm_test(eG, eQ, lag)
    print(f"{name:<52} statistic {st:+.2f}  p = {p:.4f}")
print(f"lag-1 autocorrelation of the loss differential: {ac1:.2f}  (≠ 0 ⇒ lag-0 DM understates the variance)")

**Interpretation.**

* **Folds.** Skill against persistence changes substantially from one test period to the next. A ranking from one block therefore reflects the block as much as the models. **Demonstrated.**
* **DM test.** Where the loss differential is autocorrelated, the lag-0 statistic is larger and its p-value smaller than the HAC version. This is how v5's "p < 0.001" could appear. **Demonstrated** on these errors; the size of the effect depends on the data.

**Modification (v7).** Four rolling origins × five seeds; seed-averaged hourly losses; HAC-DM with Holm correction; block-bootstrap CIs.

## 7.3 Problem P4: the seasonal ablation could not test seasonality

**Before.** v5/v6 removed `is_wet` but kept `doy_sin`/`doy_cos`.
**Evidence.** If `is_wet` can be recovered exactly from the day-of-year encoding, removing it removes no information.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
Z = df_v7[["doy_sin", "doy_cos"]].to_numpy(); w = df_v7["is_wet"].to_numpy()
acc = DecisionTreeClassifier(max_depth=6, random_state=0).fit(Z, w).score(Z, w)
print(f"🔬 is_wet recovered from (doy_sin, doy_cos) alone: accuracy = {acc:.4f}")
print("⇒ removing only is_wet leaves the season encoded in the model; the v5/v6 ablation measured noise, not seasonality.")

**Modification (v7, Part R3).** Three arms on identical samples: (A) no seasonal information; (B) calendar only; (C) calendar + `is_wet`. The v7.1 result is 📓 reproduced in §16.

## 7.4 Problem P7: conformal calibration on the early-stopping split

**Before (v5/v6).** The conformal widening was estimated on the **validation** split, which also selected the training epoch.
**Modification (v7).** A separate **calibration** split (train/val/cal/test = 60/12.5/12.5/15).
**Experiment (🔬 RECALCULATED)** on the last v7 fold: QR-GRU, MC-Dropout and a heteroscedastic MC-Dropout. The last model tests *why* MC-Dropout under-covers, i.e. whether it lacks aleatoric uncertainty.

In [ ]:
sp4 = split_chrono(X7, y7, m7, fracs=(0.60, 0.125, 0.125))
rows = []
for kind in ("QR-GRU", "MC-Dropout LSTM", "HMC-LSTM"):
    out, _ = fit_predict(kind, sp4, seed=42, passes=50)
    (pt, Qt), (_, Qv), (_, Qc) = out["test"], out["val"], out["cal"]
    yte, mte = sp4["test"][1], sp4["test"][2]
    for tag, dl in (("raw", 0.0), ("conformal on VAL (v5/v6)", cqr_delta(Qv, sp4["val"][1])),
                    ("conformal on CAL (v7)", cqr_delta(Qc, sp4["cal"][1]))):
        r = score(pt, yte, mte, Qt, dl); rows.append({"model": kind, "intervals": tag, **r})
UQ7 = pd.DataFrame(rows)
display(UQ7[["model", "intervals", "RMSE", "CRPS_q", "PICP80", "MIW"]].round(3))

**Interpretation.**

* **MC-Dropout vs HMC-LSTM.** MC-Dropout samples only weight (epistemic) uncertainty, so its raw coverage falls short of 0.80. Adding a learned noise variance (HMC-LSTM) raises coverage. This is consistent with missing aleatoric variance, and matches the full-scale v7.1 result (0.621 → 0.905, 📌). **Demonstrated** here on one fold and seed.
* **Val vs cal.** The val- and cal-based corrections usually differ modestly. The methodological point is independence: the calibration split must not have been used to select the model. That is a matter of **validity rather than a guaranteed numerical gain**, and it is reported as such.

## 7.5 Darwin record, meteorology, Singapore source (🔬 RECALCULATED where files exist)

In [ ]:
# ---- Darwin 1-min export: consistency with the 1-s record, and what it adds ----
if FILES["darwin_1min"]:
    p = FILES["darwin_1min"][0]; hdr = _header_row(p)
    cols = pd.read_csv(p, skiprows=hdr, nrows=0).columns.tolist()
    cu = next(c for c in cols if c.lower().startswith("timestamp_utc")); irr = [c for c in cols if c.lower().startswith("irradiance_mb")]
    m1 = pd.read_csv(p, skiprows=hdr, usecols=[cu] + irr, dtype={cu: str})
    tt = pd.to_datetime(m1[cu], format=_fmt(m1[cu].head(200)), errors="coerce") + pd.Timedelta(hours=9.5)
    m1g = m1[irr].apply(pd.to_numeric, errors="coerce"); m1g = m1g.mask((m1g >= 9000) | (m1g <= -999)).mean(axis=1)
    H1M = pd.DataFrame({"ts": tt.dt.floor("h"), "g": m1g}).groupby("ts").g.mean()
    ov = pd.concat([H_ACST.set_index("ts").ghi_raw, H1M], axis=1, keys=["s1", "m1"]).dropna()
    ov = ov[ov.s1 > 20]
    added = H1M.index.difference(H_ACST.ts)
    print(f"🔬 1-min vs 1-s on {len(ov):,} overlapping daylight hours: bias {100*(ov.m1.mean()/ov.s1.mean()-1):+.2f}%, "
          f"R² {np.corrcoef(ov.s1, ov.m1)[0,1]**2:.4f}")
    print(f"   hours the 1-min export adds outside the 1-s coverage: {len(added):,} ({added.min()} → {added.max()})")
    print("   ⇒ v7 extended the Darwin record; consequence: v7's test period lies entirely in the 1-min extension.")
else:
    print("Darwin 1-min export not found — record-extension evidence 📌 recorded: bias +0.00%, R² 0.9987, +16,248 h.")

# ---- Darwin on-site meteorology: present on Drive, unused in v5/v6 ----
if FILES["darwin_met"]:
    p = FILES["darwin_met"][0]; hdr = _header_row(p)
    cols = pd.read_csv(p, skiprows=hdr, nrows=0).columns.tolist()
    cu = next(c for c in cols if c.lower().startswith("timestamp_utc"))
    ta = next((c for c in cols if c.lower().startswith("airtemperature")), None)
    met = pd.read_csv(p, skiprows=hdr, usecols=[cu] + ([ta] if ta else []), dtype={cu: str})
    tt = pd.to_datetime(met[cu], format=_fmt(met[cu].head(200)), errors="coerce") + pd.Timedelta(hours=9.5)
    HM = pd.DataFrame({"ts": tt.dt.floor("h"), "t": pd.to_numeric(met[ta], errors="coerce")}).groupby("ts").t.mean()
    day = df_v7[df_v7.is_daylight].ts
    cov = HM.reindex(day).notna().mean()
    print(f"🔬 Darwin air temperature (met file) covers {100*cov:.1f}% of 2021-23 daylight hours "
          "(v5/v6 left this file unclassified and used reference-cell temperature instead).")
else:
    print("Darwin met file not found — 📌 recorded coverage (v7): temp 77%, RH 77%, wind 51%, joint 50.9%.")

# ---- Singapore: record length of the two candidate sources ----
sg = []
for key, label in (("sg_nasa", "NASA POWER hourly (v5, v7)"), ("sg_openmeteo", "Open-Meteo 10-min (v6)")):
    if FILES[key]:
        p = FILES[key][0]
        with open(p, errors="replace") as f:
            head = [next(f) for _ in range(40)]
        skip = next((i + 1 for i, l in enumerate(head) if "-END HEADER-" in l), 0)
        s = pd.read_csv(p, skiprows=skip, low_memory=False)
        if "YEAR" in s.columns:
            t = pd.to_datetime(dict(year=s.YEAR, month=s.MO, day=s.DY, hour=s.HR), errors="coerce")
        else:
            t = pd.to_datetime(s[[c for c in s.columns if "time" in c.lower()][0]], errors="coerce")
        sg.append({"source": label, "rows": len(s), "first": t.min(), "last": t.max(),
                   "span_days": (t.max() - t.min()).days, "≈ test hours at 15% (daylight ~50%)": int(len(s) * 0.15 * 0.5)})
display(pd.DataFrame(sg)) if sg else print("Singapore files not found.")

In [ ]:
print(executed_text("v7.1", "RUN-TIME DATA-DESIGN DECISIONS", 2500) if _nb("v7.1") else "")
for f in ("figV2_timing", "fig24_lookback_selection"):
    show_reproduced("v7.1", f, width=950)

### What v7 resolved
* P2: the timestamp reference.
* P3: the evaluation protocol and the statistical test.
* P4: the seasonal-ablation design.
* P7: the calibration split.
* Tuning: an equal budget per architecture.
* Darwin record: extended.
* Darwin meteorology: loaded and reported.
* Singapore source: priority restored.
* ESMAP and Yulara: rescued.
* Narrative: generated from results.

### What remained unresolved, or was introduced by v7
* **New error.** The proxy test (Part 11) compared irradiance **k_t** with **PV/capacity**. Only one of the two was clear-sky normalised (§8).
* The test period comes from the 1-min extension, while the 1-s data enter training only (stated in the report).
* Darwin met joint coverage (50.9%) was below the pre-set 60% threshold, so the exogenous experiment did not run.

---
# 8. Version 7.1 — current research implementation

**Inherited:** everything from v7, with all cached results reused. **Changed:** only Part 11, the proxy test.

## 8.1 Problem identified
v7 reported that PV was *easier* to forecast than irradiance by 18.7 pp. The two targets were normalised differently:

* irradiance was the clear-sky index k_t, with the diurnal cycle removed;
* PV was PV/capacity, with the diurnal cycle retained.

Persistence of a series that still rises and falls every day is a much weaker reference.

## 8.2 Evidence (🔬 RECALCULATED at Alice Springs)

In [ ]:
ALICE_OK = bool(FILES["alice_irr"] and FILES["alice_pv"])
if ALICE_OK:
    p = FILES["alice_irr"][0]; hdr = _header_row(p)
    cols = pd.read_csv(p, skiprows=hdr, nrows=0).columns.tolist()
    cu = next(c for c in cols if c.lower().startswith("timestamp_utc")); irr = [c for c in cols if c.lower().startswith("irradiance_mb")]
    a = pd.read_csv(p, skiprows=hdr, usecols=[cu] + irr, dtype={cu: str})
    tt = pd.to_datetime(a[cu], format=_fmt(a[cu].head(200)), errors="coerce") + pd.Timedelta(hours=9.5)
    ag = a[irr].apply(pd.to_numeric, errors="coerce"); ag = ag.mask((ag >= 9000) | (ag <= -999)).mean(axis=1)
    hb = tt.dt.floor("h")
    HA = pd.DataFrame({"ts": hb, "g": ag, "o": (tt - hb).dt.total_seconds()}).groupby("ts").agg(
        ghi_raw=("g", "mean"), n_obs=("g", "count"), ts_off_min=("o", lambda x: x.mean() / 60)).reset_index()
    pv = pd.read_csv(FILES["alice_pv"][0], usecols=lambda c: c.lower() in ("timestamp", "active_power"), low_memory=False)
    pv.columns = [c.lower() for c in pv.columns]
    pt = pd.to_datetime(pv["timestamp"], errors="coerce")
    HP = pd.DataFrame({"ts": pt.dt.floor("h"), "pv": pd.to_numeric(pv["active_power"], errors="coerce").clip(lower=0)}).groupby("ts").pv.mean()
    dA = engineer(build_version(HA, ALICE, "sample_centre", "alice"), ALICE)
    dA["pv_kw"] = HP.reindex(dA.ts).to_numpy()
    n_tr = int(len(dA) * 0.6)
    day = dA.is_daylight & (dA.solar_elev_deg > ELEV_FLOOR) & dA.pv_kw.notna()
    k_cap = np.nanpercentile((dA.pv_kw / dA.ghi_cs)[day & (dA.index < n_tr)], 99.5)
    cap = np.nanpercentile(dA.pv_kw[dA.index < n_tr], 99.5)
    dA["pv_cap"] = (dA.pv_kw / cap).clip(0, 1.2)                                    # v7 target
    dA["pv_kt"] = (dA.pv_kw / (k_cap * dA.ghi_cs.where(dA.is_daylight))).clip(0, KT_MAX)   # v7.1 target
    dA.loc[~dA.is_daylight, "pv_kt"] = 0.0
    dd = dA[dA.is_daylight & dA.kt.notna() & dA.pv_kt.notna() & dA.pv_cap.notna()].copy()
    # plain persistence y(t+1)=y(t)  vs  smart persistence y(t+1)=y(t)·GHI_cs(t+1)/GHI_cs(t)
    cs_ratio = (dA.ghi_cs / dA.ghi_cs.shift(1)).where(dA.ghi_cs.shift(1) > DAYLIGHT_CS_MIN)
    pers = []
    for col, label in (("kt", "irradiance k_t (both versions)"), ("pv_cap", "PV / capacity (v7)"),
                       ("pv_kt", "PV clear-sky index (v7.1)")):
        s_ = dA[col]; cur = s_.reindex(dd.index); prev = s_.shift(1).reindex(dd.index)
        is_index = col in ("kt", "pv_kt")                       # already clear-sky normalised
        smart = prev if is_index else prev * cs_ratio.reindex(dd.index)
        ok = prev.notna() & cur.notna() & smart.notna()
        r_plain = float(np.sqrt(((prev - cur)[ok] ** 2).mean())); r_smart = float(np.sqrt(((smart - cur)[ok] ** 2).mean()))
        pers.append({"target": label, "plain persistence RMSE": r_plain, "smart persistence RMSE": r_smart,
                     "smart better than plain by %": 100 * (1 - r_smart / r_plain)})
    display(pd.DataFrame(pers).round(3))
    print("🔬 For a clear-sky index, plain persistence IS smart persistence (0% difference by construction). "
          "For PV/capacity, the diurnal cycle alone makes plain persistence much worse than smart persistence: "
          "that is the reference v7 measured PV skill against, so PV 'skill' was inflated.")
else:
    print("Alice Springs files not found — §8 falls back to the recorded values.")

**Interpretation.** For both clear-sky indices, plain persistence already *is* smart persistence. For PV/capacity it is not: simply accounting for the sun's movement reduces persistence error by the percentage shown. v7 scored PV against this handicapped reference while irradiance was scored against the proper one, so PV's skill was inflated. v7's "PV easier by 18.7 pp" was therefore at least partly an artefact of asymmetric normalisation. **Demonstrated.**

### A second defect, inherited from v5/v6 and fixed in v7: row deletion before windowing (🔬)
v5/v6 dropped the hours where either target was missing and *then* formed 24-row windows. After the deletion, a window could span days of missing data while being treated as 24 consecutive hours.

In [ ]:
if ALICE_OK:
    both = dA.kt.notna() & dA.pv_kw.notna()
    rows_kept = dA.loc[both, "ts"].reset_index(drop=True)
    spans = (rows_kept.shift(-(LOOKBACK - 1)) - rows_kept).dt.total_seconds() / 3600
    bad = (spans > LOOKBACK - 1 + 1e-6).mean()
    print(f"🔬 v5/v6 method: {100*bad:.1f}% of 24-row windows span more than 24 real hours (they bridge deleted gaps).")
    print("   v7/v7.1 method: complete hourly grid with both targets masked → such windows are discarded by construction.")

## 8.3 Modification and 8.4 before vs after (🔬 RECALCULATED, GRU, one seed per target and definition)

In [ ]:
if ALICE_OK:
    def mirror(d, col):
        e = d.copy(); e["kt"] = e[col]
        for L in (1, 2, 3, 24): e[f"kt_lag{L}"] = e["kt"].shift(L)
        for W in (3, 24):
            r = e["kt"].rolling(W, min_periods=max(2, W // 3)); e[f"kt_roll{W}_mean"], e[f"kt_roll{W}_std"] = r.mean().shift(1), r.std().shift(1)
        return e
    gridA = dA.copy()
    badm = gridA.is_daylight & (gridA.kt.isna() | gridA.pv_kt.isna() | gridA.pv_cap.isna())
    gridA.loc[badm, ["kt", "pv_kt", "pv_cap"]] = np.nan
    gridA.loc[~gridA.is_daylight, "pv_cap"] = gridA.loc[~gridA.is_daylight, "pv_cap"].fillna(0)
    prox = []
    for col, label in (("kt", "irradiance k_t"), ("pv_cap", "PV/capacity — v7"), ("pv_kt", "PV clear-sky index — v7.1")):
        X, y, mt = build_supervised(mirror(gridA, col))
        sp = split_chrono(X, y, mt)
        out, _ = fit_predict("GRU", sp, seed=42)
        yte = sp["test"][1]; p = out["test"][0]; per = sp["test"][0][:, -1, 0]
        rm, rp = np.sqrt(np.mean((p - yte) ** 2)), np.sqrt(np.mean((per - yte) ** 2))
        prox.append({"target": label, "n_test": len(yte), "GRU RMSE": rm, "persistence RMSE": rp, "skill vs own persistence %": 100 * (1 - rm / rp)})
    PROX = pd.DataFrame(prox).round(3); display(PROX)
    s = PROX.set_index("target")["skill vs own persistence %"]
    print(f"irradiance − PV skill: v7 definition {s.iloc[0]-s.iloc[1]:+.1f} pp | v7.1 definition {s.iloc[0]-s.iloc[2]:+.1f} pp")

In [ ]:
print("📓 v7 (flawed) proxy output:"); print(executed_text("v7", "PROXY VALIDATION", 1800) if _nb("v7") else "(v7 notebook not found)")
print("\n📓 v7.1 (corrected) proxy output:"); print(executed_text("v7.1", "PROXY VALIDATION", 1800) if _nb("v7.1") else "")
show_reproduced("v7.1", "fig20_proxy_matched", width=750)

## 8.5 Research implication

With like-for-like targets, the full-scale v7.1 result (📌 −8.0 pp, significant for 3 of 4 architectures) still finds PV *more* predictable than point irradiance, but by less than v7 claimed. A physical explanation is available: the plant averages cloud shadows over its array. The proxy therefore does not *cost* skill at Alice Springs. That is a conclusion about an **arid** site only.

**What remains outstanding:** a tropical PV-power test; the Darwin met experiment, since coverage was below threshold; a full-year same-data test of the ramp-seasonality question (§11).

---
# 9. Cumulative EDA on the final data (`df_v71`, Darwin; Alice where relevant)

Every figure answers a question the research needed answered. All are 🔬 recalculated unless marked 📓.

In [ ]:
d = df_v71
print("DATASET OVERVIEW — Darwin (final data)")
display(pd.DataFrame({"rows": [len(d)], "columns": [d.shape[1]], "first": [d.ts.min()], "last": [d.ts.max()],
                      "sampling": ["1 h (aggregated from 1 s)"], "daylight hours": [int(d.is_daylight.sum())],
                      "duplicated timestamps": [int(d.ts.duplicated().sum())]}))
display(d[["ghi_raw", "ghi_wm2", "ghi_cs", "kt", "cos_zenith"]][d.is_daylight].describe().T.round(3))

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 8))
cov = d.set_index("ts").ghi_raw.notna().resample("D").mean()
ax[0, 0].plot(cov.index, 100 * cov, lw=.8); ax[0, 0].set_title("Data availability: % of hours with data, per day", fontsize=9.5, loc="left")
dl = d[d.is_daylight]
for wet, lab in ((0, "dry season"), (1, "wet season")):
    s = dl[dl.is_wet == wet].groupby(dl.ts.dt.hour).ghi_wm2.mean(); ax[0, 1].plot(s.index, s, "-o", ms=3, label=lab)
ax[0, 1].legend(); ax[0, 1].set_title("Mean diurnal GHI by season (W/m²)", fontsize=9.5, loc="left")
ax[1, 0].hist(dl.kt.dropna(), bins=60); ax[1, 0].set_title("Daylight k_t distribution (final data)", fontsize=9.5, loc="left")
mon = [dl[dl.ts.dt.month == m].kt.dropna() for m in range(1, 13)]
ax[1, 1].boxplot(mon, showfliers=False); ax[1, 1].set_xticklabels(range(1, 13))
ax[1, 1].set_title("Monthly k_t (wet season Nov–Apr is lower and more variable)", fontsize=9.5, loc="left")
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.3))
corr = dl[FEATURES].corr()
im = ax[0].imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1); ax[0].set_xticks(range(len(FEATURES))); ax[0].set_yticks(range(len(FEATURES)))
ax[0].set_xticklabels(FEATURES, rotation=90, fontsize=7); ax[0].set_yticklabels(FEATURES, fontsize=7); plt.colorbar(im, ax=ax[0])
ax[0].set_title("Feature correlation (daylight hours)", fontsize=9.5, loc="left")
s = d.kt.where(d.is_daylight)
ac = [s.autocorr(k) for k in range(1, 49)]
ax[1].bar(range(1, 49), ac); ax[1].set_xlabel("lag (h)"); ax[1].set_title("Autocorrelation of k_t: why persistence is hard at t+1", fontsize=9.5, loc="left")
plt.tight_layout(); plt.show()
tgt = d.kt.shift(-1)
print("Feature–target correlation (target = k_t at t+1, daylight):")
display(dl[FEATURES].corrwith(tgt[d.is_daylight]).sort_values(ascending=False).round(3).to_frame("r"))

### 9.1 Ramps: the v5 definition vs the elevation-floored definition, and the regime-share defect

In [ ]:
rows = []
for v, dv in (("v5", df_v5), ("v6", df_v6), ("v7/v7.1", df_v71)):
    day = dv[dv.is_daylight]
    for wet, lab in ((1, "wet"), (0, "dry")):
        s = day[day.is_wet == wet]
        rows.append({"data": v, "season": lab, "ramp % (v5 definition)": 100 * s.is_ramp_raw.mean(),
                     "ramp % (elevation > 15°)": 100 * s.is_ramp_kt.mean()})
display(pd.DataFrame(rows).round(2))
day = df_v71[df_v71.is_daylight]
reg = pd.cut(day.kt, [-0.01, 0.3, 0.7, 2], labels=["Overcast", "Broken", "Clear"])
print("Sky-regime shares — v6 method (denominator = all daylight hours) vs v7 (hours with valid k_t):")
display(pd.DataFrame({"v6 method %": 100 * reg.value_counts() / len(day),
                      "v7 method %": 100 * reg.value_counts() / reg.notna().sum()}).round(2).T.assign(total=lambda x: x.sum(axis=1)))

**Observation.** The v5 ramp definition fires at dawn and dusk, where a small absolute change divided by a tiny clear-sky value becomes a large Δk_t. The elevation floor removes these. The v6 regime shares did not sum to 100% because missing-k_t hours stayed in the denominator. **Both defects are demonstrated.**

Whether Darwin's *seasonal ordering* of ramps depends on timing can be read from the table above (v5/v6 vs v7 on the same 2021–23 data). This is the same-data test the report said was still outstanding.

In [ ]:
if FILES["darwin_1min"]:
    p = FILES["darwin_1min"][0]; hdr = _header_row(p)
    cols = pd.read_csv(p, skiprows=hdr, nrows=0).columns.tolist()
    cu = next(c for c in cols if c.lower().startswith("timestamp_utc")); irr = [c for c in cols if c.lower().startswith("irradiance_mb")]
    m1 = pd.read_csv(p, skiprows=hdr, usecols=[cu] + irr, dtype={cu: str})
    t1 = pd.to_datetime(m1[cu], format=_fmt(m1[cu].head(200)), errors="coerce") + pd.Timedelta(hours=9.5)
    g1 = m1[irr].apply(pd.to_numeric, errors="coerce").mean(axis=1)
    f1 = pd.DataFrame({"t": t1, "g": g1}).dropna().set_index("t").sort_index()
    f1 = f1[f1.index.year == 2022]
    cs1 = clearsky_haurwitz(solar_geometry(f1.index, DARWIN["lat"], DARWIN["lon"], 9.5)); f1["cs"] = cs1
    def vi(fr, dtm):
        fr = fr[fr.cs > 20]
        return np.sum(np.sqrt(np.diff(fr.g) ** 2 + dtm ** 2)) / np.sum(np.sqrt(np.diff(fr.cs) ** 2 + dtm ** 2))
    step_min = float(pd.Series(f1.index).diff().dt.total_seconds().median() / 60) if len(f1) > 1 else 1.0
    per_day = 1440 / max(step_min, 1e-6)
    rows = []
    for day_, fr in f1.groupby(f1.index.floor("D")):
        if len(fr) > 0.4 * per_day:
            hr = fr.resample("h").mean()
            rows.append({"day": day_, "VI native": vi(fr, step_min), "VI hourly": vi(hr.dropna(), 60.0)})
    VI = pd.DataFrame(rows, columns=["day", "VI native", "VI hourly"])
    print(f"native sampling of the export: {step_min:.1f} min; days analysed: {len(VI)}")
    if len(VI):
        display(VI[["VI native", "VI hourly"]].describe().round(2))
    else:
        print("no complete days in 2022 in this file — VI comparison skipped")
    print("🔬 At native (1-min) resolution the variability index separates calm from variable days; at hourly resolution it "
          "collapses towards ~1 — why v7 replaced it with the within-day SD of hourly Δk_t.")

In [ ]:
for f in ("fig07_crosssite", "fig09_proxy", "fig16_energy_regimes"):
    show_reproduced("v7.1", f, width=950)

---
# 10. Cumulative feature engineering: evolution and justification

| Feature | v5 | v6 | v7 | v7.1 | Reason for change | Research implication |
|---|---|---|---|---|---|---|
| k_t (target and level) | local bins, sun @HH:00 | sun @bin centre | **UTC bins**, sun @centre | = v7 | timing P1, P2 | the target itself changed (§6–7) |
| kt_lag1/2/3/24 | ✓ | ✓ | ✓ | ✓ | — | inherit k_t's timing |
| kt_roll3/24 mean, SD (shifted) | ✓ | ✓ | ✓ | ✓ | — | past-only by construction (§12) |
| cos_zenith, ghi_cs_norm | @HH:00 | @centre | @centre, correct clock | = v7 | timing | geometry inputs corrected |
| hour_sin/cos | local label | local label | ACST label | = v7 | timing | label now means Darwin time |
| doy_sin/cos | ✓ | ✓ | ✓ | ✓ | — | carries the season (see is_wet) |
| is_wet | ✓ | ✓ | ✓ (+ 3-arm test) | = v7 | ablation confound P4 | redundant given doy (§7.3); its value was tested properly |
| temp_c | **cell** temperature; screened out | same | air temperature from the met file; excluded by design | = v7 | wrong variable; transfer comparability | endogenous vector keeps zero-shot sites evaluable |
| rh_pct, wind_ms | absent at Darwin | absent | loaded; excluded by design | = v7 | coverage 51% joint | exogenous value not testable (below threshold) |
| is_ramp (diagnostic, not an input) | Δk_t only | + elevation > 15° | same | same | dawn/dusk artefacts | ramp results now describe weather |
| *_obs covariates | — | — | observed-only copies (no climatology fill) | = v7 | leakage (§12) | safe exogenous use |
| PV target (Alice) | PV/cap | PV/cap | PV/cap | **PV clear-sky index** | normalisation asymmetry | proxy test valid only in v7.1 |

Nothing from v5's feature set was removed without a reason. The final 16-feature vector is v5's screened vector, recomputed on correctly timed data.

In [ ]:
fe_cmp = pd.DataFrame({v: dv[dv.is_daylight][["kt", "kt_lag1", "kt_roll3_mean", "cos_zenith", "hour_sin"]].mean()
                       for v, dv in (("v5", df_v5), ("v6", df_v6), ("v7/v7.1", df_v71))})
print("🔬 Daylight means of selected features under each version (same raw data):")
display(fe_cmp.round(4))
display(screen(df_v71, FEATURES).assign(version="v7/v7.1 final vector"))

---
# 11. Timestamp and temporal-integrity investigation (detailed; consolidated from the v8 comparison notebook)

**What.** The decisive physical test uses the **raw 1-second samples** of July 2022 (dry season, mostly clear), with no binning and no clear-sky model.

On a clear day, irradiance is symmetric about solar noon, which is fixed by longitude and the equation of time. So the centre of the measured curve must fall on solar noon **in the correct time basis**.

In [ ]:
if len(JULY_1S):
    fmtj = _fmt(JULY_1S.raw_utc.head(200))
    tu = pd.to_datetime(JULY_1S.raw_utc, format=fmtj); tl = pd.to_datetime(JULY_1S.raw_local, format=fmtj)
    timestamp_v5 = timestamp_v6 = tl; timestamp_v7 = timestamp_v71 = tu + pd.Timedelta(hours=9.5)
    g = JULY_1S.ghi.to_numpy()
    def eot(doy):
        b = 2 * np.pi * (np.asarray(doy, float) - 1) / 365
        return 229.18 * (0.000075 + 0.001868*np.cos(b) - 0.032077*np.sin(b) - 0.014615*np.cos(2*b) - 0.040849*np.sin(2*b))
    dayk = timestamp_v7.dt.normalize()
    STEP_S = float(tu.diff().dt.total_seconds().median())
    STEP_1MIN = max(1, int(round(60 / STEP_S)))                 # samples per minute (1 for 1-min data, 60 for 1-s)
    print(f"native sampling of the July rows: {STEP_S:.0f} s")
    stats_ = []
    for dday, idx in pd.Series(np.arange(len(g))).groupby(dayk.values):
        gi = g[idx.values]
        if np.nanmax(gi) < 500: continue
        stats_.append({"day": dday, "rough": np.nanmean(np.abs(np.diff(gi[::STEP_1MIN]))) / np.nanmax(gi)})
    clear = pd.DataFrame(stats_).nsmallest(10, "rough").day
    rows = []
    for dday in clear:
        sel = (dayk == dday).to_numpy()
        noon = (720 - 4 * DARWIN["lon"] - eot(pd.DatetimeIndex([dday]).dayofyear)[0] + 60 * 9.5) / 60
        w = np.clip(np.nan_to_num(g[sel]), 0, None)
        c = lambda t: float(((t.dt.hour + t.dt.minute / 60 + t.dt.second / 3600) * w).sum() / w.sum())
        rows.append({"day": dday.date(), "v5/v6 (Local) − noon, min": 60 * (c(timestamp_v5[sel]) - noon),
                     "v7/v7.1 (UTC+9:30) − noon, min": 60 * (c(timestamp_v7[sel]) - noon)})
    NOON = pd.DataFrame(rows); display(NOON.round(2)); display(NOON.iloc[:, 1:].agg(["mean", "std"]).round(2))
    sel = (dayk == clear.iloc[0]).to_numpy()
    grid = pd.date_range(clear.iloc[0] + pd.Timedelta(hours=5), clear.iloc[0] + pd.Timedelta(hours=21), freq="1min")
    fig, ax = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
    for a, t, ttl, col in ((ax[0], timestamp_v5, "v5/v6: GHI vs Timestamp_Local", "#c1440e"), (ax[1], timestamp_v7, "v7/v7.1: GHI vs UTC+9:30 (ACST)", "#2a4d8f")):
        a.plot(t[sel].iloc[::STEP_1MIN], g[sel][::STEP_1MIN], color=col, lw=1, label="measured (1-s)")
        a.plot(grid, clearsky_haurwitz(solar_geometry(grid, DARWIN["lat"], DARWIN["lon"], 9.5)), "k--", lw=1, label="clear-sky model")
        a.set_title(ttl, fontsize=9.5, loc="left"); a.legend(fontsize=8)
        a.xaxis.set_major_formatter(matplotlib.dates.DateFormatter("%H:%M"))
    plt.tight_layout(); plt.show()
else:
    print("July 2022 1-s rows unavailable in this window; see the v8 timestamp notebook (📌 −30.8 ± 0.8 vs −0.8 ± 0.8 min).")

### 11.1 Consequences of the timestamp treatment: Demonstrated vs Potential

| Consequence | Classification | Evidence |
|---|---|---|
| Solar noon and peak read ~30 min early (v5/v6 labels) | **Demonstrated** | §11 table and figure |
| Hourly GHI values differ (different half-hour windows) | **Demonstrated** | §7.1 (bins), §5.1 (same samples) |
| Clear-day k_t tilted; more hours removed by QC as non-physical | **Demonstrated** | §6.4, §7.1 |
| Forecast results (persistence RMSE, skill) shift | **Demonstrated** (controlled experiment A) | §6.4.1, §7.1 |
| Sunrise/sunset appear early by the same 30 min | **Demonstrated** | left panel of §11 figure |
| Irradiance mis-paired with correctly timed sources (e.g. Alice 241 PV export) | **Potential** (supported by the v7 alignment log; not recomputed here) | 📌 v7 align_log |
| Feature–target misalignment *within* the Darwin series | **Not supported**: all columns shift together; the bias is against the sun | §12 |
| Train/test boundaries moved | **Not supported**: a 30-min relabelling moves nothing across a boundary materially | §12 |
| Calendar date of daylight observations changed | **Not supported**: midnight is always dark | — |

---
# 12. Data leakage and methodological validation (Problem → Evidence → Correction → Validation)

Each check runs on the final data and the final functions.

In [ ]:
checks = []
# 1. rolling statistics use only the past
e = df_v71.copy(); t = e.kt.to_numpy()
i = int(np.flatnonzero(np.isfinite(e.kt_roll3_mean.to_numpy()))[500])
checks.append(("Rolling mean excludes the current hour", np.isclose(e.kt_roll3_mean.iloc[i], np.nanmean(t[i-3:i])),
               "kt_roll3_mean(t) == mean(kt[t-3..t-1])"))
# 2. lag construction
checks.append(("kt_lag1(t) == kt(t-1)", np.allclose(e.kt_lag1.iloc[1:].to_numpy(), e.kt.iloc[:-1].to_numpy(), equal_nan=True), "shift(1)"))
# 3. target strictly after window
X, y, mt = build_supervised(df_v71)
checks.append(("Target timestamp is after every input", True, "by construction: target index = window end + horizon"))
# 4. chronological split, no overlap
sp_ = split_chrono(X, y, mt)
ordered = sp_["train"][2].ts.max() < sp_["val"][2].ts.min() < sp_["test"][2].ts.min()
checks.append(("Train < val < test in time", bool(ordered), "chronological, never shuffled"))
# 5. scaler fitted on train only
sc = Standardiser().fit(sp_["train"][0])
checks.append(("Scaler statistics from training rows only", np.allclose(sc.m, sp_["train"][0].reshape(-1, X.shape[-1]).mean(0)), "Standardiser.fit(train)"))
# 6. duplicated timestamps
checks.append(("No duplicated hourly timestamps", int(df_v71.ts.duplicated().sum()) == 0, "hourly grid"))
# 7. interpolation leakage share
share = 100 * sp_["test"][2].kt_imputed.mean()
checks.append((f"Interpolated test targets = {share:.1f}% (use the value after the gap)", share < 10,
               "v7 R5(a): skill unchanged when excluded (📌)"))
display(pd.DataFrame(checks, columns=["check", "passed", "evidence"]))

| Issue | Problem | Evidence | Correction | Validation |
|---|---|---|---|---|
| Covariate imputation by whole-record climatology (v5–v7 `impute`) | Uses test-period values to fill training gaps | Inspection of `impute()` | Harmless while the covariates are screened out; v7 adds *_obs copies (no climatology fill) for any model that uses them | Covariates absent from the final vector (§10) |
| Conformal on the validation split (v5/v6) | The same data chose the epoch and the correction | §7.4 | Separate calibration split (v7) | §7.4 table |
| Seasonal ablation (v5/v6) | `is_wet` fully recoverable from doy | §7.3 (accuracy printed) | Three-arm design | 📓 R3 in §16 |
| DM without HAC (v5/v6) | Autocorrelated loss differential | §7.2 | Newey–West + Holm | §7.2 |
| Row deletion before windowing (proxy, v5/v6) | Windows bridge missing days | §8.2 | Masked full grid | §8.2 |
| Tuning on data later tested | Would leak selection into evaluation | v7 design | Tuning on fold-1 validation, which precedes every test block | 📓 Fig 24 / 8C |
| Timestamp reference (v5/v6) | Target biased against the sun | §6, §7, §11 | UTC + 9:30 | clear-day gate −1.3 min (📌) |

---
# 13. Version comparison matrix

| Component | v5 | v6 | v7 | v7.1 | Why changed | Evidence | Final decision |
|---|---|---|---|---|---|---|---|
| Darwin dataset | 1-s, 2021–23 | same | + same-station 1-min fill → 2025 | = v7 | statistical power | §7.5 consistency | v7 (test period in 1-min data, stated) |
| Timestamp column | Local | Local | **UTC + 9:30** | = v7 | Local = UTC+9 | §5.2, §11 | v7 |
| Geometry instant | HH:00 | bin centre | bin centre | = v7 | hourly mean ≈ HH:30 | §6 | v6 onward |
| Timing validation | none | lag ±45 min | clear-day gate ±10 min | = v7 | −31 min passed in v6 | 📌 | v7 |
| Cleaning / QC | ✓ | ✓ | ✓ + sentinel mask before averaging | = v7 | Yulara 99,999.9 | 📌 | v7 |
| Missing values | k_t ≤3 h interp. | same | same + leakage sensitivity | = v7 | leakage check | §12 | v7 |
| EDA | Figs 1–10, 5A | same | VI replaced, shares fixed, domain shift | = v7 | defects | §9 | v7 |
| Features | 19 → screen (cell temp) | same | endogenous by design; air temp as *_obs | = v7 | wrong variable; transfer | §10 | v7 |
| Target | k_t (mistimed) | k_t (half-corrected) | k_t (correct) | = v7 | timing | §6–7 | v7 |
| Split | 70/15/15 | 70/15/15 | 60/12.5/12.5/15 + 4 rolling origins | = v7 | fragility | §7.2 | v7 |
| Scaling | train-only | same | same | same | — | §12 | unchanged |
| Baselines | 3 naive | 3 naive | per fold; best-naive by horizon | = v7 | weak reference | 📓 R5 | v7 |
| Deterministic models | LSTM, GRU | same | same, tuned | = v7 | equal budget | 📓 8C | v7 |
| Probabilistic models | MC-D, QR-GRU | same | + HMC-LSTM (diagnostic) | = v7 | epistemic vs aleatoric | §7.4 | v7 |
| Calibration | conformal on val | same | conformal on cal | = v7 | independence | §7.4 | v7 |
| Statistical tests | DM, no HAC | same | HAC-DM + Holm + bootstrap CIs | = v7 | anti-conservative | §7.2 | v7 |
| Proxy test | row deletion | same | full grid; PV/cap (**new error**) | **PV clear-sky index** | normalisation | §8 | v7.1 |
| Singapore | NASA POWER, 14 y | **Open-Meteo, 1 month** | NASA POWER (priority rule) | = v7 | record length | §7.5 | v7 (v5 was better than v6 here) |
| Narrative | hand-written | stale numbers, contradictions | generated from results | = v7 | integrity | 📌 | v7 |
| Reproducibility | seeds, checkpoints | + fingerprints | + per-unit cache, results registry | = v7 | resumability, audit | — | v7 |

# 14. Research development decision log

| Version | Issue / observation | Evidence | Decision | Modification | Result | Research impact |
|---|---|---|---|---|---|---|
| v5→v6 | Clear-sky plot shows two branches | v5 Fig 8; §5.5 | Correct the geometry instant | sun at mean sample time | tilt halved; skill 21.8% → 7.0% (📌) | v5 headline partly an artefact |
| v6 | Need row-level checks | no validation in v5 | Add Part 3A | lag, coverage, units | lag −31 min measured but **passed** | tolerance too loose (v6 weakness) |
| v6 | Singapore NASA POWER "missing" | misreported; file present | switched source | Open-Meteo 1 month | 51 test hours | **regression** |
| v6→v7 | Local column ≠ ACST | offset 9.0 h; solar noon −30.8 min | Use the UTC column | UTC + 9:30; clear-day gate | lag −1.3 min | target correct |
| v6→v7 | Seed SD ≈ architecture spread | v6 12A | Robust protocol | 4 folds × 5 seeds, HAC-DM, CIs | three models tied (📌) | ranking claims bounded |
| v6→v7 | Ablation confounded | §7.3 | Three arms | R3 | is_wet null for 3/4 (📌) | O3 answered validly |
| v6→v7 | Calibration reused val | design review | cal split | 4-way split | honest coverage | valid O2/O4 |
| v6→v7 | Unequal tuning | config effect > architecture spread | Equal budget | 8C grid | tuned per model | fair comparison |
| v6→v7 | Small Darwin sample | 535 test windows | Extend record | 1-min fill if consistent | 4,887 test hours | power; test period in 1-min data |
| v7→v7.1 | "PV easier by 18.7 pp" | asymmetric normalisation (§8.2) | Like-for-like targets | PV clear-sky index | −8.0 pp (📌) | valid proxy conclusion |

# 15. Consolidated final methodology

| Stage | What is done | Why | Introduced / modified | Evidence |
|---|---|---|---|---|
| Raw data | DKASC 1-s reference cells (+ same-station 1-min fill) | research-grade tropical ground data | v5; fill v7 | §5.1, §7.5 |
| Validation | coverage, gaps, units, sentinels, clear-day timing gate | errors are silent otherwise | v6; gate v7 | §5.2 |
| Timestamp handling | UTC column + 9:30; sun at mean sample time | Local = UTC+9; hourly mean ≈ HH:30 | v6 (instant), v7 (reference) | §6, §7, §11 |
| Cleaning | range/QC, non-physical flag, night zero | physical plausibility | v5 | §5.4 |
| Target | k_t = GHI/GHI_cs, ≤3 h gaps interpolated | removes astronomy | v5; correct from v7 | §6–7 |
| EDA | availability, diurnal, distributions, correlation, ACF, regimes, ramps (elev > 15°) | problem characterisation | v5; fixes v6–v7 | §9 |
| Features | 16 endogenous, past-only | available at every site; no leakage | v5; design v7 | §10, §12 |
| Dataset construction | 24-h windows, daylight targets, complete-window rule | no invented weather | v5 | §5.7 |
| Split | train/val/cal/test + 4 rolling origins | independence; robustness | v7 | §7.2 |
| Models | LSTM, GRU, MC-D LSTM, QR-GRU (+ HMC diagnostic), equal-budget tuning | RQ; fairness | v5; v7 | §7.4 |
| UQ | quantiles / MC passes; CQR on the cal split | coverage validity | v5; v7 | §7.4 |
| Evaluation | RMSE, CRPS, PICP, MIW, Winkler; skill vs persistence and best naive; HAC-DM + Holm; bootstrap CIs | proper scores; valid inference | v5; v7 | §7.2 |
| Proxy | matched grid, like-for-like clear-sky indices | valid O1b | v7.1 | §8 |
| Interpretation | findings generated from results; Demonstrated vs Potential | integrity | v7 | §16–19 |

---
# 16. Final results and evidence (📓 REPRODUCED from executed v7.1; 🔬 controlled experiments above)

In [ ]:
print(executed_text("v7.1", "CONSOLIDATED FINDINGS", 7000) if _nb("v7.1") else "executed v7.1 notebook not found")

In [ ]:
for f in ("figR1_skill_ci", "figR2_fold_skill", "figR3_calibration", "figR6_wet_dry_coverage",
          "figR4_seasonal_ablation", "figR5_horizon", "fig19_ramp_extreme", "fig14_cross_climate",
          "fig15_decision", "fig22_reserve"):
    show_reproduced("v7.1", f, width=950)

In [ ]:
print("🔬 CONTROLLED EXPERIMENTS RUN IN THIS NOTEBOOK (small scale, for attribution — not the headline results)")
display(summ_A)
display(FOLDS7.round(2))
display(UQ7[["model", "intervals", "PICP80", "MIW"]].round(3))
if ALICE_OK: display(PROX)

---
# 17. Research interpretation

| Initially thought (v5) | What the data actually showed (v7.1) |
|---|---|
| QR-GRU clearly best (+21.8% over persistence) | QR-GRU, GRU and LSTM are **tied** in point accuracy (+11–12%, overlapping CIs). QR-GRU is distinguished by **calibration** (80% coverage 0.805, stable across seasons) at no accuracy cost |
| MC-Dropout is a usable UQ method | It under-covers (0.62; 0.45 in the wet season) because it lacks aleatoric uncertainty (HMC-LSTM check) |
| A wet/dry flag may help | Null for 3 of 4 architectures; the recent k_t history already carries the season |
| Ramps are moderately harder (+15%) | Ramp-hour RMSE is +121–173%, and coverage collapses there (QR-GRU 0.46) |
| Irradiance proxy costs 6.5 pp | On like-for-like targets, PV is *more* predictable than point irradiance at Alice Springs (arid) |
| Timestamps were correct | The Local column is UTC+9. Two timing errors existed; v6 fixed one, v7 the other |

# 18. What could have been misinterpreted

| Misinterpretation | Version | Class | Evidence |
|---|---|---|---|
| "QR-GRU beats persistence by ~22%" | v5 | **Demonstrated** as timing-sensitive | §6.4.1, 📌 |
| "Clear mornings show cloud enhancement" | v5/v6 | **Demonstrated** artefact | §6.4, §11 |
| "Solar peak occurs ~12:20 ACST" | v5/v6 | **Demonstrated** 30-min shift | §11 |
| "Architectures rank X > Y" from one block | v5/v6 | **Demonstrated** fragility | §7.2 |
| "Seasonal flag has effect Z" | v5/v6 | **Demonstrated** invalid design | §7.3 |
| "PV is 18.7 pp easier than irradiance" | v7 | **Demonstrated** artefact | §8 |
| "Equatorial behaviour is shown" from 51 hours | v6 | **Demonstrated** insufficient sample | §7.5 |
| Irradiance–PV pairing offset at Alice | v5/v6 | **Potential** | 📌 v7 alignment log |

# 19. What the corrections now allow us to conclude
* The timing of the modelling data is verified against the sun (−1.3 min).
* Architecture comparisons carry uncertainty, and only differences beyond it are claimed.
* Calibration conclusions rest on an independent calibration split.
* The seasonal-conditioning conclusion rests on a valid three-arm design.
* The proxy conclusion rests on like-for-like targets.

# 20. Remaining limitations
* One training site; the test period comes from the 1-min extension.
* Hourly resolution only; the sub-hourly study is not run.
* The proxy was tested at an arid site.
* Equatorial evidence is zero-shot, from satellite and one short ground record.
* Darwin meteorology joint coverage is below threshold.
* The Haurwitz clear-sky model sits about 6% low on clear days, a level bias.
* Controlled experiments here are single- or two-seed and for attribution only.

# 21. Research traceability to the proposal and PDM

| Requirement | v5 | v6 | v7 | v7.1 | Evidence generated | Final status | Report section |
|---|---|---|---|---|---|---|---|
| O1 accuracy vs persistence | single block | single block | R1–R2 with CIs | = | Fig R1–R2, §6.4.1 | Answered (three tied) | 6.1 |
| O1b proxy | row deletion | same | invalid normalisation | **valid** | §8 | Answered (PV more predictable, arid) | 6.6 |
| O2 calibration / trade-off | val-conformal | same | cal-conformal, HMC | = | Fig R3, R6, §7.4 | Answered | 6.2 |
| O3 seasonal / ramps | confounded | confounded | 3-arm; floored ramps | = | Fig R4, §7.3, §9.1 | Answered (null for 3/4) | 6.3 |
| O4 decision support | raw | raw | conformal, equal reliability | = | Fig 15, 22 | Answered (negative) | 6.4 |
| PDM O8 cross-climate | 4 sites | 7 (Singapore weak) | 8 + shift | = | Fig 14 | Answered | 6.7 |
| PDM O6 monsoon phase | — | — | — | — | — | Not achieved | 8 |
| Data validity (timing) | unverified | partial | verified | = | §11 | Resolved | 3.3 |

# 22. Final academic conclusion

The research evolved because each version exposed evidence that the previous one could not see:

* **v5** built a complete pipeline whose headline result was partly a timing artefact.
* **v6** corrected half of that artefact, added validation, and regressed the Singapore arm.
* **v7** found the second timing error from physical evidence, rebuilt the evaluation so that conclusions carry uncertainty, and repaired the ablation, calibration and tuning designs. It also introduced a proxy-normalisation error.
* **v7.1** corrected that error.

The final methodology is more defensible not because it contains more code, but because every decision in it traces to evidence shown in this notebook.

# 23. Reproducibility information

In [ ]:
print(f"Python {sys.version.split()[0]} | {platform.platform()}")
print(f"pandas {pd.__version__} | numpy {np.__version__} | tensorflow {tf.__version__} | matplotlib {matplotlib.__version__}")
print(f"seed {SEED}; Darwin files: {[os.path.basename(p) for p in FILES['darwin_1s']]}")
print(f"cache folder: {CACHE}")
print("Evidence labels: 🔬 recalculated here | 📓 reproduced from executed notebooks | 📌 recorded values with provenance")